In [2]:
from pathlib import Path
import gc
import json

import duckdb
import numpy as np
import pandas as pd

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "packages/network-security-ai/pyproject.toml").is_file())

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SPLIT_V2_DIR = (
    PROJECT_ROOT
    / "data/processed/network_ml_split_v2"
)

TRAIN_V2 = SPLIT_V2_DIR / "train.parquet"
VALID_V2 = SPLIT_V2_DIR / "valid.parquet"

V3_DIR = (
    PROJECT_ROOT
    / "data/processed/network_ml_dev_v3/model_sample"
)

TRAIN_V3 = V3_DIR / "train_v3_sample.parquet"
VALID_V3 = V3_DIR / "valid_v3_sample.parquet"

FEATURE_PATH = (
    PROJECT_ROOT
    / "results/feature_selection/top40_features.csv"
)

V4_DIR = (
    PROJECT_ROOT
    / "data/processed/network_ml_dev_v4"
)

RESULT_DIR = (
    PROJECT_ROOT
    / "results/model_sample_v4"
)

TEMP_DIR = (
    PROJECT_ROOT
    / "data/tmp/duckdb_v4"
)

V4_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN V2    :", TRAIN_V2)
print("VALID V2    :", VALID_V2)
print("TRAIN V3    :", TRAIN_V3)
print("VALID V3    :", VALID_V3)

PROJECT_ROOT: /home/geonug/kdt-linux/ML
TRAIN V2    : /home/geonug/kdt-linux/ML/data/processed/network_ml_split_v2/train.parquet
VALID V2    : /home/geonug/kdt-linux/ML/data/processed/network_ml_split_v2/valid.parquet
TRAIN V3    : /home/geonug/kdt-linux/ML/data/processed/network_ml_dev_v3/model_sample/train_v3_sample.parquet
VALID V3    : /home/geonug/kdt-linux/ML/data/processed/network_ml_dev_v3/model_sample/valid_v3_sample.parquet


In [3]:
required_files = [
    TRAIN_V2,
    VALID_V2,
    TRAIN_V3,
    VALID_V3,
    FEATURE_PATH,
]

for path in required_files:
    print(
        f"{path.name:<30}",
        "OK" if path.exists() else "MISSING",
    )

    assert path.exists(), path

print("\n모든 입력 파일 확인 완료")

train.parquet                  OK
valid.parquet                  OK
train_v3_sample.parquet        OK
valid_v3_sample.parquet        OK
top40_features.csv             OK

모든 입력 파일 확인 완료


In [4]:
con = duckdb.connect()

con.execute("SET memory_limit='2GB'")
con.execute("SET threads=2")
con.execute(
    f"SET temp_directory='{TEMP_DIR}'"
)
con.execute(
    "SET preserve_insertion_order=false"
)

print(
    con.execute(
        "SELECT version()"
    ).fetchone()
)

print("DuckDB safety settings applied.")

('v1.5.6',)
DuckDB safety settings applied.


In [5]:
train_v2_distribution = con.execute(
    f"""
    SELECT
        label_id,
        COUNT(*) AS row_count
    FROM read_parquet('{TRAIN_V2}')
    GROUP BY label_id
    ORDER BY label_id
    """
).df()

train_v2_total = int(
    train_v2_distribution[
        "row_count"
    ].sum()
)

train_v2_distribution["ratio"] = (
    train_v2_distribution["row_count"]
    / train_v2_total
)

print("Train V2 total:", f"{train_v2_total:,}")

print(
    train_v2_distribution.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

Train V2 total: 11,416,137
 label_id  row_count    ratio
        0    9451891 0.827941
        1     194856 0.017068
        2        428 0.000037
        3        166 0.000015
        4     475783 0.041676
        5       1203 0.000105
        6     402962 0.035298
        7      29158 0.002554
        8     324733 0.028445
        9      85458 0.007486
       10       7766 0.000680
       11     173083 0.015161
       12     113111 0.009908
       13         55 0.000005
       14     155484 0.013620


In [6]:
valid_v2_distribution = con.execute(
    f"""
    SELECT
        label_id,
        COUNT(*) AS row_count
    FROM read_parquet('{VALID_V2}')
    GROUP BY label_id
    ORDER BY label_id
    """
).df()

valid_v2_total = int(
    valid_v2_distribution[
        "row_count"
    ].sum()
)

valid_v2_distribution["ratio"] = (
    valid_v2_distribution["row_count"]
    / valid_v2_total
)

print("Valid V2 total:", f"{valid_v2_total:,}")

print(
    valid_v2_distribution.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

Valid V2 total: 2,390,002
 label_id  row_count    ratio
        0    2012773 0.842164
        1      43368 0.018146
        2         98 0.000041
        3         28 0.000012
        4     104966 0.043919
        5        268 0.000112
        6      86583 0.036227
        7       6160 0.002577
        8      69024 0.028880
        9       6882 0.002879
       10       1611 0.000674
       11      19005 0.007952
       12      24623 0.010303
       13         13 0.000005
       14      14600 0.006109


In [7]:
train_v3_distribution = con.execute(
    f"""
    SELECT
        label_id,
        COUNT(*) AS row_count
    FROM read_parquet('{TRAIN_V3}')
    GROUP BY label_id
    ORDER BY label_id
    """
).df()

train_v3_total = int(
    train_v3_distribution[
        "row_count"
    ].sum()
)

train_v3_distribution["ratio"] = (
    train_v3_distribution["row_count"]
    / train_v3_total
)

print("Train V3 total:", f"{train_v3_total:,}")

print(
    train_v3_distribution.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

Train V3 total: 702,217
 label_id  row_count    ratio
        0     100000 0.142406
        1     100000 0.142406
        2        394 0.000561
        3        165 0.000235
        4     100000 0.142406
        5       1203 0.001713
        6     100000 0.142406
        7      29090 0.041426
        8     100000 0.142406
        9         44 0.000063
       10       6874 0.009789
       11         39 0.000056
       12      98237 0.139896
       13         54 0.000077
       14      66117 0.094155


In [8]:
distribution_compare = (
    train_v2_distribution[
        ["label_id", "row_count", "ratio"]
    ]
    .rename(
        columns={
            "row_count": "v2_count",
            "ratio": "v2_ratio",
        }
    )
    .merge(
        train_v3_distribution[
            ["label_id", "row_count", "ratio"]
        ].rename(
            columns={
                "row_count": "v3_count",
                "ratio": "v3_ratio",
            }
        ),
        on="label_id",
        how="outer",
    )
    .fillna(0)
)

distribution_compare[
    "ratio_change"
] = (
    distribution_compare["v3_ratio"]
    - distribution_compare["v2_ratio"]
)

distribution_compare[
    "representation_multiplier"
] = np.divide(
    distribution_compare["v3_ratio"],
    distribution_compare["v2_ratio"],
    out=np.zeros(
        len(distribution_compare),
        dtype=float,
    ),
    where=(
        distribution_compare[
            "v2_ratio"
        ].to_numpy()
        > 0
    ),
)

print(
    distribution_compare.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

 label_id  v2_count  v2_ratio  v3_count  v3_ratio  ratio_change  representation_multiplier
        0   9451891  0.827941    100000  0.142406     -0.685535                   0.172000
        1    194856  0.017068    100000  0.142406      0.125338                   8.343227
        2       428  0.000037       394  0.000561      0.000524                  14.965812
        3       166  0.000015       165  0.000235      0.000220                  16.159343
        4    475783  0.041676    100000  0.142406      0.100730                   3.416952
        5      1203  0.000105      1203  0.001713      0.001608                  16.257278
        6    402962  0.035298    100000  0.142406      0.107109                   4.034444
        7     29158  0.002554     29090  0.041426      0.038872                  16.219364
        8    324733  0.028445    100000  0.142406      0.113961                   5.006352
        9     85458  0.007486        44  0.000063     -0.007423                   0.008370

In [9]:
train_v2_distribution.to_csv(
    RESULT_DIR / "train_v2_distribution.csv",
    index=False,
)

valid_v2_distribution.to_csv(
    RESULT_DIR / "valid_v2_distribution.csv",
    index=False,
)

train_v3_distribution.to_csv(
    RESULT_DIR / "train_v3_distribution.csv",
    index=False,
)

distribution_compare.to_csv(
    RESULT_DIR / "v2_v3_distribution_compare.csv",
    index=False,
)

print("Distribution diagnostics saved.")

Distribution diagnostics saved.


In [10]:
V3_ROOT = (
    PROJECT_ROOT
    / "data/processed/network_ml_dev_v3"
)

print("V3 ROOT:", V3_ROOT)
print("Exists :", V3_ROOT.exists())

print("\n=== V3 files ===")

for path in sorted(V3_ROOT.rglob("*")):

    if path.is_file():

        print(
            f"{path.relative_to(V3_ROOT)} "
            f"({path.stat().st_size / 1024**2:.2f} MB)"
        )

V3 ROOT: /home/geonug/kdt-linux/ML/data/processed/network_ml_dev_v3
Exists : True

=== V3 files ===
model_sample/train_v3_sample.parquet (58.35 MB)
model_sample/valid_v3_sample.parquet (13.00 MB)
train_parts/class_00.parquet (925.94 MB)
train_parts/class_01.parquet (5.06 MB)
train_parts/class_02.parquet (0.06 MB)
train_parts/class_03.parquet (0.04 MB)
train_parts/class_04.parquet (9.55 MB)
train_parts/class_05.parquet (0.12 MB)
train_parts/class_06.parquet (25.15 MB)
train_parts/class_07.parquet (2.87 MB)
train_parts/class_08.parquet (5.89 MB)
train_parts/class_09.parquet (0.02 MB)
train_parts/class_10.parquet (0.69 MB)
train_parts/class_11.parquet (0.02 MB)
train_parts/class_12.parquet (9.33 MB)
train_parts/class_13.parquet (0.02 MB)
train_parts/class_14.parquet (5.82 MB)
valid_parts/class_00.parquet (188.22 MB)
valid_parts/class_01.parquet (1.20 MB)
valid_parts/class_02.parquet (0.03 MB)
valid_parts/class_03.parquet (0.02 MB)
valid_parts/class_04.parquet (2.12 MB)
valid_parts/class_0

In [11]:
import pyarrow.parquet as pq

parquet_paths = sorted(
    V3_ROOT.rglob("*.parquet")
)

print(
    "Parquet files:",
    len(parquet_paths)
)

for path in parquet_paths[:30]:

    pf = pq.ParquetFile(path)

    print(
        "\nFILE:",
        path.relative_to(V3_ROOT)
    )

    print(
        "rows:",
        f"{pf.metadata.num_rows:,}"
    )

    print(
        "columns:",
        pf.schema.names
    )

Parquet files: 32

FILE: model_sample/train_v3_sample.parquet
rows: 702,217
columns: ['Dst Port', 'Protocol', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Si

In [12]:
MODEL_EXCLUDE_COLUMNS = {
    "Label",
    "label_id",
    "source_file",
    "Timestamp",
    "hour",
    "day_of_week",
}

schema_df = con.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{TRAIN_V2}')
    """
).df()

all_columns = schema_df[
    "column_name"
].tolist()

MODEL_FEATURES = [
    col
    for col in all_columns
    if col not in MODEL_EXCLUDE_COLUMNS
]

print("All columns    :", len(all_columns))
print("Model features :", len(MODEL_FEATURES))

assert len(MODEL_FEATURES) == 78

for i, feature in enumerate(
    MODEL_FEATURES,
    1,
):
    print(f"{i:02d}. {feature}")

All columns    : 84
Model features : 78
01. Dst Port
02. Protocol
03. Flow Duration
04. Tot Fwd Pkts
05. Tot Bwd Pkts
06. TotLen Fwd Pkts
07. TotLen Bwd Pkts
08. Fwd Pkt Len Max
09. Fwd Pkt Len Min
10. Fwd Pkt Len Mean
11. Fwd Pkt Len Std
12. Bwd Pkt Len Max
13. Bwd Pkt Len Min
14. Bwd Pkt Len Mean
15. Bwd Pkt Len Std
16. Flow Byts/s
17. Flow Pkts/s
18. Flow IAT Mean
19. Flow IAT Std
20. Flow IAT Max
21. Flow IAT Min
22. Fwd IAT Tot
23. Fwd IAT Mean
24. Fwd IAT Std
25. Fwd IAT Max
26. Fwd IAT Min
27. Bwd IAT Tot
28. Bwd IAT Mean
29. Bwd IAT Std
30. Bwd IAT Max
31. Bwd IAT Min
32. Fwd PSH Flags
33. Bwd PSH Flags
34. Fwd URG Flags
35. Bwd URG Flags
36. Fwd Header Len
37. Bwd Header Len
38. Fwd Pkts/s
39. Bwd Pkts/s
40. Pkt Len Min
41. Pkt Len Max
42. Pkt Len Mean
43. Pkt Len Std
44. Pkt Len Var
45. FIN Flag Cnt
46. SYN Flag Cnt
47. RST Flag Cnt
48. PSH Flag Cnt
49. ACK Flag Cnt
50. URG Flag Cnt
51. CWE Flag Count
52. ECE Flag Cnt
53. Down/Up Ratio
54. Pkt Size Avg
55. Fwd Seg Size Avg
56

In [13]:
FREQ_DIR = (
    V4_DIR
    / "frequency_parts"
)

FREQ_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Frequency checkpoint:")
print(FREQ_DIR)

Frequency checkpoint:
/home/geonug/kdt-linux/ML/data/processed/network_ml_dev_v4/frequency_parts


In [14]:
def quote_identifier(name):
    escaped = name.replace(
        '"',
        '""',
    )
    return f'"{escaped}"'


FEATURE_SQL = ",\n        ".join(
    quote_identifier(col)
    for col in MODEL_FEATURES
)

print(
    "Feature count:",
    len(MODEL_FEATURES),
)

Feature count: 78


In [15]:
TEST_CLASS_ID = 13

test_output = (
    FREQ_DIR
    / f"class_{TEST_CLASS_ID:02d}.parquet"
)

print(test_output)

/home/geonug/kdt-linux/ML/data/processed/network_ml_dev_v4/frequency_parts/class_13.parquet


In [16]:
query = f"""
COPY (
    SELECT
        {FEATURE_SQL},
        {TEST_CLASS_ID}::SMALLINT
            AS label_id,
        COUNT(*)::BIGINT
            AS frequency
    FROM read_parquet('{TRAIN_V2}')
    WHERE label_id = {TEST_CLASS_ID}
    GROUP BY
        {FEATURE_SQL}
)
TO '{test_output}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
)
"""

con.execute(query)

print("Created:", test_output)

Created: /home/geonug/kdt-linux/ML/data/processed/network_ml_dev_v4/frequency_parts/class_13.parquet


In [17]:
test_freq = pd.read_parquet(
    test_output
)

print(
    test_freq[
        ["label_id", "frequency"]
    ].to_string(
        index=False
    )
)

print()
print(
    "Unique patterns:",
    f"{len(test_freq):,}"
)

print(
    "Frequency sum :",
    f"{test_freq['frequency'].sum():,}"
)

print(
    "Original rows :",
    "55"
)

assert (
    int(test_freq["frequency"].sum())
    == 55
)

del test_freq
gc.collect()

 label_id  frequency
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          2
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13          1
       13    

5

In [18]:
train_counts = dict(
    zip(
        train_v2_distribution[
            "label_id"
        ].astype(int),
        train_v2_distribution[
            "row_count"
        ].astype(int),
    )
)

train_counts

{0: 9451891,
 1: 194856,
 2: 428,
 3: 166,
 4: 475783,
 5: 1203,
 6: 402962,
 7: 29158,
 8: 324733,
 9: 85458,
 10: 7766,
 11: 173083,
 12: 113111,
 13: 55,
 14: 155484}

In [19]:
import time

for class_id in range(15):

    output_path = (
        FREQ_DIR
        / f"class_{class_id:02d}.parquet"
    )

    # 이미 만들어진 checkpoint는 건너뜀
    if output_path.exists():

        print(
            f"[SKIP] class={class_id:02d} "
            f"{output_path.name}"
        )

        continue

    expected_rows = train_counts[
        class_id
    ]

    print(
        f"\n[START] class={class_id:02d} "
        f"original={expected_rows:,}"
    )

    start = time.perf_counter()

    query = f"""
    COPY (
        SELECT
            {FEATURE_SQL},
            {class_id}::SMALLINT
                AS label_id,
            COUNT(*)::BIGINT
                AS frequency
        FROM read_parquet('{TRAIN_V2}')
        WHERE label_id = {class_id}
        GROUP BY
            {FEATURE_SQL}
    )
    TO '{output_path}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """

    con.execute(query)

    elapsed = (
        time.perf_counter()
        - start
    )

    print(
        f"[DONE] class={class_id:02d} "
        f"time={elapsed:.1f}s "
        f"size="
        f"{output_path.stat().st_size / 1024**2:.2f} MB"
    )

    gc.collect()


[START] class=00 original=9,451,891


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[DONE] class=00 time=43.6s size=898.10 MB

[START] class=01 original=194,856
[DONE] class=01 time=0.6s size=4.80 MB

[START] class=02 original=428
[DONE] class=02 time=0.1s size=0.06 MB

[START] class=03 original=166
[DONE] class=03 time=0.1s size=0.03 MB

[START] class=04 original=475,783
[DONE] class=04 time=0.6s size=9.60 MB

[START] class=05 original=1,203
[DONE] class=05 time=0.1s size=0.11 MB

[START] class=06 original=402,962
[DONE] class=06 time=0.9s size=24.70 MB

[START] class=07 original=29,158
[DONE] class=07 time=0.3s size=2.85 MB

[START] class=08 original=324,733
[DONE] class=08 time=0.6s size=5.91 MB

[START] class=09 original=85,458
[DONE] class=09 time=0.3s size=0.02 MB

[START] class=10 original=7,766
[DONE] class=10 time=0.1s size=0.68 MB

[START] class=11 original=173,083
[DONE] class=11 time=0.2s size=0.02 MB

[START] class=12 original=113,111
[DONE] class=12 time=0.7s size=9.15 MB
[SKIP] class=13 class_13.parquet

[START] class=14 original=155,484
[DONE] class=14

In [20]:
integrity_rows = []

for class_id in range(15):

    path = (
        FREQ_DIR
        / f"class_{class_id:02d}.parquet"
    )

    assert path.exists(), path

    result = con.execute(
        f"""
        SELECT
            COUNT(*) AS unique_patterns,
            SUM(frequency) AS reconstructed_rows,
            MIN(frequency) AS min_frequency,
            MAX(frequency) AS max_frequency,
            AVG(frequency) AS avg_frequency
        FROM read_parquet('{path}')
        """
    ).fetchone()

    (
        unique_patterns,
        reconstructed_rows,
        min_frequency,
        max_frequency,
        avg_frequency,
    ) = result

    original_rows = (
        train_counts[class_id]
    )

    integrity_rows.append(
        {
            "label_id": class_id,
            "original_rows": original_rows,
            "unique_patterns": unique_patterns,
            "reconstructed_rows": (
                reconstructed_rows
            ),
            "duplicate_rows": (
                original_rows
                - unique_patterns
            ),
            "compression_ratio": (
                unique_patterns
                / original_rows
            ),
            "min_frequency": (
                min_frequency
            ),
            "max_frequency": (
                max_frequency
            ),
            "avg_frequency": (
                avg_frequency
            ),
            "match": (
                original_rows
                == reconstructed_rows
            ),
        }
    )


integrity_df = pd.DataFrame(
    integrity_rows
)

print(
    integrity_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

 label_id  original_rows  unique_patterns  reconstructed_rows  duplicate_rows  compression_ratio  min_frequency  max_frequency  avg_frequency  match
        0        9451891          7445818             9451891         2006073           0.787760              1           7487       1.269423   True
        1         194856           100976              194856           93880           0.518208              1           1630       1.929726   True
        2            428              394                 428              34           0.920561              1              5       1.086294   True
        3            166              165                 166               1           0.993976              1              2       1.006061   True
        4         475783           139130              475783          336653           0.292423              1            162       3.419701   True
        5           1203             1203                1203               0           1.000000          

In [21]:
assert integrity_df[
    "match"
].all(), (
    "Frequency reconstruction mismatch"
)

original_total = int(
    integrity_df[
        "original_rows"
    ].sum()
)

reconstructed_total = int(
    integrity_df[
        "reconstructed_rows"
    ].sum()
)

print(
    "Original Train V2 :",
    f"{original_total:,}",
)

print(
    "Reconstructed     :",
    f"{reconstructed_total:,}",
)

print(
    "Match             :",
    original_total
    == reconstructed_total,
)

assert (
    original_total
    == 11_416_137
)

assert (
    reconstructed_total
    == original_total
)

Original Train V2 : 11,416,137
Reconstructed     : 11,416,137
Match             : True


In [22]:
integrity_df.to_csv(
    RESULT_DIR
    / "frequency_checkpoint_integrity.csv",
    index=False,
)

print(
    "Saved:",
    RESULT_DIR
    / "frequency_checkpoint_integrity.csv"
)

Saved: /home/geonug/kdt-linux/ML/results/model_sample_v4/frequency_checkpoint_integrity.csv


In [23]:
weight_summary_rows = []

for class_id in range(15):

    path = (
        FREQ_DIR
        / f"class_{class_id:02d}.parquet"
    )

    result = con.execute(
        f"""
        SELECT
            COUNT(*) AS unique_patterns,

            SUM(
                1.0
            ) AS weight_unique,

            SUM(
                LN(1.0 + frequency)
            ) AS weight_log1p,

            SUM(
                SQRT(frequency)
            ) AS weight_sqrt,

            SUM(
                frequency
            ) AS weight_frequency

        FROM read_parquet('{path}')
        """
    ).fetchone()

    (
        unique_patterns,
        weight_unique,
        weight_log1p,
        weight_sqrt,
        weight_frequency,
    ) = result

    weight_summary_rows.append(
        {
            "label_id": class_id,
            "unique_patterns": unique_patterns,
            "weight_unique": weight_unique,
            "weight_log1p": weight_log1p,
            "weight_sqrt": weight_sqrt,
            "weight_frequency": weight_frequency,
        }
    )


weight_summary = pd.DataFrame(
    weight_summary_rows
)

print(
    weight_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}",
    )
)

 label_id  unique_patterns weight_unique  weight_log1p  weight_sqrt  weight_frequency
        0          7445818     7445818.0   5476072.962  7841927.175           9451891
        1           100976      100976.0     71961.636   106081.543            194856
        2              394         394.0       285.656      407.084               428
        3              165         165.0       114.775      165.414               166
        4           139130      139130.0    133448.303   194053.409            475783
        5             1203        1203.0       833.856     1203.000              1203
        6           402392      402392.0    279147.878   402628.005            402962
        7            29090       29090.0     20190.988    29117.974             29158
        8           101779      101779.0    107743.112   149539.842            324733
        9               44          44.0       208.625      948.437             85458
       10             6874        6874.0      4856.986

In [24]:
weight_columns = [
    "weight_unique",
    "weight_log1p",
    "weight_sqrt",
    "weight_frequency",
]

for col in weight_columns:

    total = weight_summary[col].sum()

    ratio_col = (
        col.replace(
            "weight_",
            "ratio_",
        )
    )

    weight_summary[ratio_col] = (
        weight_summary[col]
        / total
    )


display_columns = [
    "label_id",
    "unique_patterns",
    "ratio_unique",
    "ratio_log1p",
    "ratio_sqrt",
    "ratio_frequency",
]

print(
    weight_summary[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

 label_id  unique_patterns                        ratio_unique  ratio_log1p  ratio_sqrt  ratio_frequency
        0          7445818      0.8872189213175105977947435701     0.881362    0.880728         0.827941
        1           100976     0.01203196449321712538809329300     0.011582    0.011914         0.017068
        2              394  0.00004694773025597713717030539379     0.000046    0.000046         0.000037
        3              165  0.00001966085150313763358654921314     0.000018    0.000019         0.000015
        4           139130     0.01657826830079720582361570923     0.021478    0.021794         0.041676
        5             1203   0.0001433454809592398376037497176     0.000134    0.000135         0.000105
        6           402392     0.04794769307909429487368915741     0.044928    0.045219         0.035298
        7            29090    0.003466267698341053097167979455     0.003250    0.003270         0.002554
        8           101779     0.0121276473038657285382

In [25]:
FOCUS_CLASS_IDS = [
    0,   # Benign
    9,   # SlowHTTPTest
    11,  # FTP-BruteForce
    12,  # Infilteration
]

focus_weight = weight_summary[
    weight_summary[
        "label_id"
    ].isin(FOCUS_CLASS_IDS)
][
    [
        "label_id",
        "unique_patterns",
        "ratio_unique",
        "ratio_log1p",
        "ratio_sqrt",
        "ratio_frequency",
    ]
].copy()

print(
    focus_weight.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

 label_id  unique_patterns                        ratio_unique  ratio_log1p  ratio_sqrt  ratio_frequency
        0          7445818      0.8872189213175105977947435701     0.881362    0.880728         0.827941
        9               44 0.000005242893734170035623079790170     0.000034    0.000107         0.007486
       11               39 0.000004647110355287077029547995832     0.000030    0.000129         0.015161
       12            98237     0.01170559435826504067055657607     0.011613    0.011546         0.009908


In [26]:
def effective_ratio(
    df,
    numerator_id,
    denominator_id,
    column,
):
    numerator = float(
        df.loc[
            df["label_id"] == numerator_id,
            column,
        ].iloc[0]
    )

    denominator = float(
        df.loc[
            df["label_id"] == denominator_id,
            column,
        ].iloc[0]
    )

    return numerator / denominator


print(
    "=== Benign : Infilteration ==="
)

for column in [
    "weight_unique",
    "weight_log1p",
    "weight_sqrt",
    "weight_frequency",
]:

    ratio = effective_ratio(
        weight_summary,
        0,
        12,
        column,
    )

    print(
        f"{column:<18}: "
        f"{ratio:.2f}:1"
    )

=== Benign : Infilteration ===
weight_unique     : 75.79:1
weight_log1p      : 75.90:1
weight_sqrt       : 76.28:1
weight_frequency  : 83.56:1


In [27]:
print(
    "=== SlowHTTPTest : FTP-BruteForce ==="
)

for column in [
    "weight_unique",
    "weight_log1p",
    "weight_sqrt",
    "weight_frequency",
]:

    ratio = effective_ratio(
        weight_summary,
        9,
        11,
        column,
    )

    print(
        f"{column:<18}: "
        f"{ratio:.4f}:1"
    )

=== SlowHTTPTest : FTP-BruteForce ===
weight_unique     : 1.1282:1
weight_log1p      : 1.1293:1
weight_sqrt       : 0.8257:1
weight_frequency  : 0.4937:1


In [28]:
JOIN_CONDITION = "\n        AND ".join(
    f'a.{quote_identifier(col)} '
    f'IS NOT DISTINCT FROM '
    f'b.{quote_identifier(col)}'
    for col in MODEL_FEATURES
)

print("Join feature count:", len(MODEL_FEATURES))

Join feature count: 78


In [29]:
def analyze_conflict_pair(
    class_a,
    class_b,
):
    path_a = (
        FREQ_DIR
        / f"class_{class_a:02d}.parquet"
    )

    path_b = (
        FREQ_DIR
        / f"class_{class_b:02d}.parquet"
    )

    query = f"""
    SELECT
        COUNT(*) AS conflict_patterns,

        COALESCE(
            SUM(a.frequency),
            0
        ) AS rows_class_a,

        COALESCE(
            SUM(b.frequency),
            0
        ) AS rows_class_b,

        COALESCE(
            SUM(
                a.frequency
                + b.frequency
            ),
            0
        ) AS total_conflict_rows,

        COALESCE(
            SUM(
                LEAST(
                    a.frequency,
                    b.frequency
                )
            ),
            0
        ) AS unavoidable_minority_rows,

        MAX(a.frequency)
            AS max_frequency_a,

        MAX(b.frequency)
            AS max_frequency_b

    FROM read_parquet('{path_a}') a

    INNER JOIN read_parquet('{path_b}') b
        ON {JOIN_CONDITION}
    """

    result = con.execute(
        query
    ).fetchone()

    return {
        "class_a": class_a,
        "class_b": class_b,
        "conflict_patterns": int(
            result[0] or 0
        ),
        "rows_class_a": int(
            result[1] or 0
        ),
        "rows_class_b": int(
            result[2] or 0
        ),
        "total_conflict_rows": int(
            result[3] or 0
        ),
        "unavoidable_minority_rows": int(
            result[4] or 0
        ),
        "max_frequency_a": (
            int(result[5])
            if result[5] is not None
            else 0
        ),
        "max_frequency_b": (
            int(result[6])
            if result[6] is not None
            else 0
        ),
    }

In [30]:
slow_ftp = analyze_conflict_pair(
    9,
    11,
)

print(
    json.dumps(
        slow_ftp,
        indent=2,
    )
)

{
  "class_a": 9,
  "class_b": 11,
  "conflict_patterns": 10,
  "rows_class_a": 71831,
  "rows_class_b": 157005,
  "total_conflict_rows": 228836,
  "unavoidable_minority_rows": 66638,
  "max_frequency_a": 62024,
  "max_frequency_b": 87297
}


In [31]:
slow_path = (
    FREQ_DIR
    / "class_09.parquet"
)

ftp_path = (
    FREQ_DIR
    / "class_11.parquet"
)

slow_ftp_detail = con.execute(
    f"""
    SELECT
        a.frequency
            AS slowhttp_frequency,

        b.frequency
            AS ftp_frequency,

        a.frequency
            + b.frequency
            AS total_frequency,

        CASE
            WHEN a.frequency
                 > b.frequency
                THEN 'SlowHTTPTest'

            WHEN b.frequency
                 > a.frequency
                THEN 'FTP-BruteForce'

            ELSE 'Tie'
        END AS majority_label

    FROM read_parquet('{slow_path}') a

    INNER JOIN read_parquet('{ftp_path}') b
        ON {JOIN_CONDITION}

    ORDER BY
        total_frequency DESC
    """
).df()

print(
    slow_ftp_detail.to_string(
        index=False
    )
)

 slowhttp_frequency  ftp_frequency  total_frequency majority_label
              62024          87297           149321 FTP-BruteForce
               1028          64336            65364 FTP-BruteForce
               5493           1092             6585   SlowHTTPTest
               1932           2975             4907 FTP-BruteForce
                831            127              958   SlowHTTPTest
                 85            595              680 FTP-BruteForce
                207            440              647 FTP-BruteForce
                207            134              341   SlowHTTPTest
                 20              7               27   SlowHTTPTest
                  4              2                6   SlowHTTPTest


In [32]:
import time

start = time.perf_counter()

benign_infil = analyze_conflict_pair(
    0,
    12,
)

elapsed = (
    time.perf_counter()
    - start
)

print(
    json.dumps(
        benign_infil,
        indent=2,
    )
)

print(
    f"\nElapsed: {elapsed:.1f}s"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{
  "class_a": 0,
  "class_b": 12,
  "conflict_patterns": 15493,
  "rows_class_a": 90450,
  "rows_class_b": 28904,
  "total_conflict_rows": 119354,
  "unavoidable_minority_rows": 28237,
  "max_frequency_a": 605,
  "max_frequency_b": 30
}

Elapsed: 2.9s


In [33]:
pair_summary = pd.DataFrame([
    {
        "pair": "SlowHTTPTest / FTP-BruteForce",
        **slow_ftp,
    },
    {
        "pair": "Benign / Infilteration",
        **benign_infil,
    },
])

pair_summary[
    "class_a_conflict_ratio"
] = (
    pair_summary["rows_class_a"]
    / pair_summary["class_a"].map(
        train_counts
    )
)

pair_summary[
    "class_b_conflict_ratio"
] = (
    pair_summary["rows_class_b"]
    / pair_summary["class_b"].map(
        train_counts
    )
)

pair_summary[
    "ambiguity_ratio"
] = (
    pair_summary[
        "unavoidable_minority_rows"
    ]
    / pair_summary[
        "total_conflict_rows"
    ]
)

print(
    pair_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

                         pair  class_a  class_b  conflict_patterns  rows_class_a  rows_class_b  total_conflict_rows  unavoidable_minority_rows  max_frequency_a  max_frequency_b  class_a_conflict_ratio  class_b_conflict_ratio  ambiguity_ratio
SlowHTTPTest / FTP-BruteForce        9       11                 10         71831        157005               228836                      66638            62024            87297                0.840542                0.907108         0.291204
       Benign / Infilteration        0       12              15493         90450         28904               119354                      28237              605               30                0.009570                0.255537         0.236582


In [35]:
top40_df = pd.read_csv(FEATURE_PATH)

print("columns:", top40_df.columns.tolist())
print()
print(top40_df.head(10).to_string(index=False))
print()
print(top40_df.dtypes)

columns: ['rank', 'feature']

 rank           feature
    1 Fwd Act Data Pkts
    2  Fwd Seg Size Min
    3    Fwd Header Len
    4      ECE Flag Cnt
    5 Init Fwd Win Byts
    6      URG Flag Cnt
    7      RST Flag Cnt
    8          Dst Port
    9     Fwd PSH Flags
   10        Fwd Pkts/s

rank       int64
feature      str
dtype: object


In [36]:
top40_df = pd.read_csv(FEATURE_PATH)

TOP40_FEATURES = (
    top40_df
    .sort_values("rank")
    ["feature"]
    .astype(str)
    .tolist()
)

assert len(TOP40_FEATURES) == 40
assert TOP40_FEATURES[0] == "Fwd Act Data Pkts"

missing_features = [
    feature
    for feature in TOP40_FEATURES
    if feature not in MODEL_FEATURES
]

assert not missing_features, (
    f"Train V2에 없는 feature: {missing_features}"
)

TOP40_SQL = ",\n        ".join(
    quote_identifier(feature)
    for feature in TOP40_FEATURES
)

V4_TOP40_DIR = (
    V4_DIR
    / "top40_frequency_parts"
)

V4_TOP40_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Top40 features:", len(TOP40_FEATURES))
print("Output:", V4_TOP40_DIR)

print("\n=== Top40 Feature Order ===")

for i, feature in enumerate(
    TOP40_FEATURES,
    1,
):
    print(f"{i:02d}. {feature}")

Top40 features: 40
Output: /home/geonug/kdt-linux/ML/data/processed/network_ml_dev_v4/top40_frequency_parts

=== Top40 Feature Order ===
01. Fwd Act Data Pkts
02. Fwd Seg Size Min
03. Fwd Header Len
04. ECE Flag Cnt
05. Init Fwd Win Byts
06. URG Flag Cnt
07. RST Flag Cnt
08. Dst Port
09. Fwd PSH Flags
10. Fwd Pkts/s
11. Fwd Pkt Len Mean
12. Pkt Len Max
13. SYN Flag Cnt
14. Fwd Seg Size Avg
15. Bwd Header Len
16. Init Bwd Win Byts
17. Fwd IAT Std
18. Fwd IAT Max
19. Pkt Size Avg
20. Fwd IAT Tot
21. Bwd Pkt Len Max
22. PSH Flag Cnt
23. Flow IAT Mean
24. Bwd IAT Min
25. Bwd Pkt Len Std
26. Pkt Len Mean
27. Bwd IAT Max
28. Fwd Pkt Len Std
29. ACK Flag Cnt
30. TotLen Bwd Pkts
31. Pkt Len Std
32. FIN Flag Cnt
33. Bwd IAT Mean
34. Flow Duration
35. Flow IAT Max
36. Subflow Bwd Byts
37. Idle Max
38. Fwd URG Flags
39. Bwd Pkts/s
40. Bwd IAT Std


In [37]:
for class_id in range(15):

    source = (
        FREQ_DIR
        / f"class_{class_id:02d}.parquet"
    )

    output = (
        V4_TOP40_DIR
        / f"class_{class_id:02d}.parquet"
    )

    if output.exists():
        print(
            f"[SKIP] class={class_id:02d}"
        )
        continue

    con.execute(
        f"""
        COPY (
            SELECT
                {TOP40_SQL},
                label_id,
                frequency
            FROM read_parquet('{source}')
        )
        TO '{output}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )

    print(
        f"[DONE] class={class_id:02d} "
        f"{output.stat().st_size / 1024**2:.2f} MB"
    )

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[DONE] class=00 524.31 MB
[DONE] class=01 2.98 MB
[DONE] class=02 0.03 MB
[DONE] class=03 0.02 MB
[DONE] class=04 5.73 MB
[DONE] class=05 0.06 MB
[DONE] class=06 14.54 MB
[DONE] class=07 1.67 MB
[DONE] class=08 3.49 MB
[DONE] class=09 0.01 MB
[DONE] class=10 0.34 MB
[DONE] class=11 0.01 MB
[DONE] class=12 5.26 MB
[DONE] class=13 0.01 MB
[DONE] class=14 3.65 MB


In [38]:
total_size = 0

for path in sorted(
    V4_TOP40_DIR.glob("class_*.parquet")
):
    size_mb = (
        path.stat().st_size
        / 1024**2
    )

    total_size += size_mb

    print(
        f"{path.name:<20} "
        f"{size_mb:>10.2f} MB"
    )

print(
    "\nTotal:",
    f"{total_size:.2f} MB"
)

class_00.parquet         524.31 MB
class_01.parquet           2.98 MB
class_02.parquet           0.03 MB
class_03.parquet           0.02 MB
class_04.parquet           5.73 MB
class_05.parquet           0.06 MB
class_06.parquet          14.54 MB
class_07.parquet           1.67 MB
class_08.parquet           3.49 MB
class_09.parquet           0.01 MB
class_10.parquet           0.34 MB
class_11.parquet           0.01 MB
class_12.parquet           5.26 MB
class_13.parquet           0.01 MB
class_14.parquet           3.65 MB

Total: 562.11 MB


In [39]:
TOP40_AGG_DIR = (
    V4_DIR
    / "top40_aggregated_parts"
)

TOP40_AGG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(TOP40_AGG_DIR)

/home/geonug/kdt-linux/ML/data/processed/network_ml_dev_v4/top40_aggregated_parts


In [40]:
import time
import gc

for class_id in range(15):

    source = (
        V4_TOP40_DIR
        / f"class_{class_id:02d}.parquet"
    )

    output = (
        TOP40_AGG_DIR
        / f"class_{class_id:02d}.parquet"
    )

    if output.exists():
        print(
            f"[SKIP] class={class_id:02d}"
        )
        continue

    print(
        f"\n[START] class={class_id:02d}"
    )

    start = time.perf_counter()

    con.execute(
        f"""
        COPY (
            SELECT
                {TOP40_SQL},

                {class_id}::SMALLINT
                    AS label_id,

                SUM(frequency)::BIGINT
                    AS frequency

            FROM read_parquet('{source}')

            GROUP BY
                {TOP40_SQL}
        )
        TO '{output}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )

    elapsed = (
        time.perf_counter()
        - start
    )

    print(
        f"[DONE] class={class_id:02d} "
        f"time={elapsed:.1f}s "
        f"size="
        f"{output.stat().st_size / 1024**2:.2f} MB"
    )

    gc.collect()


[START] class=00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[DONE] class=00 time=18.1s size=534.95 MB

[START] class=01
[DONE] class=01 time=0.2s size=2.98 MB

[START] class=02
[DONE] class=02 time=0.0s size=0.03 MB

[START] class=03
[DONE] class=03 time=0.0s size=0.02 MB

[START] class=04
[DONE] class=04 time=0.2s size=5.78 MB

[START] class=05
[DONE] class=05 time=0.0s size=0.06 MB

[START] class=06
[DONE] class=06 time=0.4s size=14.76 MB

[START] class=07
[DONE] class=07 time=0.2s size=1.70 MB

[START] class=08
[DONE] class=08 time=0.2s size=3.54 MB

[START] class=09
[DONE] class=09 time=0.0s size=0.01 MB

[START] class=10
[DONE] class=10 time=0.0s size=0.34 MB

[START] class=11
[DONE] class=11 time=0.0s size=0.01 MB

[START] class=12
[DONE] class=12 time=0.2s size=5.31 MB

[START] class=13
[DONE] class=13 time=0.0s size=0.01 MB

[START] class=14
[DONE] class=14 time=0.1s size=3.66 MB


In [41]:
import time
import gc

for class_id in range(15):

    source = (
        V4_TOP40_DIR
        / f"class_{class_id:02d}.parquet"
    )

    output = (
        TOP40_AGG_DIR
        / f"class_{class_id:02d}.parquet"
    )

    if output.exists():
        print(
            f"[SKIP] class={class_id:02d}"
        )
        continue

    print(
        f"\n[START] class={class_id:02d}"
    )

    start = time.perf_counter()

    con.execute(
        f"""
        COPY (
            SELECT
                {TOP40_SQL},

                {class_id}::SMALLINT
                    AS label_id,

                SUM(frequency)::BIGINT
                    AS frequency

            FROM read_parquet('{source}')

            GROUP BY
                {TOP40_SQL}
        )
        TO '{output}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )

    elapsed = (
        time.perf_counter()
        - start
    )

    print(
        f"[DONE] class={class_id:02d} "
        f"time={elapsed:.1f}s "
        f"size="
        f"{output.stat().st_size / 1024**2:.2f} MB"
    )

    gc.collect()

[SKIP] class=00
[SKIP] class=01
[SKIP] class=02
[SKIP] class=03
[SKIP] class=04
[SKIP] class=05
[SKIP] class=06
[SKIP] class=07
[SKIP] class=08
[SKIP] class=09
[SKIP] class=10
[SKIP] class=11
[SKIP] class=12
[SKIP] class=13
[SKIP] class=14


In [42]:
top40_integrity_rows = []

for class_id in range(15):

    original_path = (
        V4_TOP40_DIR
        / f"class_{class_id:02d}.parquet"
    )

    aggregated_path = (
        TOP40_AGG_DIR
        / f"class_{class_id:02d}.parquet"
    )

    before = con.execute(
        f"""
        SELECT
            COUNT(*),
            SUM(frequency)
        FROM read_parquet('{original_path}')
        """
    ).fetchone()

    after = con.execute(
        f"""
        SELECT
            COUNT(*),
            SUM(frequency),
            MAX(frequency),
            AVG(frequency)
        FROM read_parquet('{aggregated_path}')
        """
    ).fetchone()

    top40_integrity_rows.append({
        "label_id": class_id,

        "patterns_78": int(before[0]),
        "rows_before": int(before[1]),

        "patterns_top40": int(after[0]),
        "rows_after": int(after[1]),

        "patterns_merged": (
            int(before[0])
            - int(after[0])
        ),

        "pattern_retention": (
            int(after[0])
            / int(before[0])
        ),

        "max_frequency_top40": int(
            after[2]
        ),

        "avg_frequency_top40": float(
            after[3]
        ),

        "frequency_match": (
            int(before[1])
            == int(after[1])
        ),
    })


top40_integrity_df = pd.DataFrame(
    top40_integrity_rows
)

print(
    top40_integrity_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

 label_id  patterns_78  rows_before  patterns_top40  rows_after  patterns_merged  pattern_retention  max_frequency_top40  avg_frequency_top40  frequency_match
        0      7445818      9451891         7445262     9451891              556           0.999925                 7487             1.269518             True
        1       100976       194856          100976      194856                0           1.000000                 1630             1.929726             True
        2          394          428             394         428                0           1.000000                    5             1.086294             True
        3          165          166             165         166                0           1.000000                    2             1.006061             True
        4       139130       475783          139130      475783                0           1.000000                  162             3.419701             True
        5         1203         1203           

In [43]:
assert top40_integrity_df[
    "frequency_match"
].all()

assert (
    top40_integrity_df[
        "rows_after"
    ].sum()
    == 11_416_137
)

print(
    "Reconstructed rows:",
    f"{top40_integrity_df['rows_after'].sum():,}"
)

print(
    "All frequency matches:",
    top40_integrity_df[
        "frequency_match"
    ].all()
)

top40_integrity_df.to_csv(
    RESULT_DIR
    / "top40_projection_integrity.csv",
    index=False,
)

Reconstructed rows: 11,416,137
All frequency matches: True


In [44]:
TOP40_JOIN_CONDITION = "\n        AND ".join(
    f'a.{quote_identifier(col)} '
    f'IS NOT DISTINCT FROM '
    f'b.{quote_identifier(col)}'
    for col in TOP40_FEATURES
)


def analyze_top40_conflict_pair(
    class_a,
    class_b,
):

    path_a = (
        TOP40_AGG_DIR
        / f"class_{class_a:02d}.parquet"
    )

    path_b = (
        TOP40_AGG_DIR
        / f"class_{class_b:02d}.parquet"
    )

    result = con.execute(
        f"""
        SELECT
            COUNT(*) AS conflict_patterns,

            COALESCE(
                SUM(a.frequency),
                0
            ) AS rows_class_a,

            COALESCE(
                SUM(b.frequency),
                0
            ) AS rows_class_b,

            COALESCE(
                SUM(
                    a.frequency
                    + b.frequency
                ),
                0
            ) AS total_conflict_rows,

            COALESCE(
                SUM(
                    LEAST(
                        a.frequency,
                        b.frequency
                    )
                ),
                0
            ) AS unavoidable_minority_rows,

            MAX(a.frequency)
                AS max_frequency_a,

            MAX(b.frequency)
                AS max_frequency_b

        FROM read_parquet('{path_a}') a

        INNER JOIN read_parquet('{path_b}') b
            ON {TOP40_JOIN_CONDITION}
        """
    ).fetchone()

    return {
        "class_a": class_a,
        "class_b": class_b,

        "conflict_patterns": int(
            result[0] or 0
        ),

        "rows_class_a": int(
            result[1] or 0
        ),

        "rows_class_b": int(
            result[2] or 0
        ),

        "total_conflict_rows": int(
            result[3] or 0
        ),

        "unavoidable_minority_rows": int(
            result[4] or 0
        ),

        "max_frequency_a": int(
            result[5] or 0
        ),

        "max_frequency_b": int(
            result[6] or 0
        ),
    }

In [45]:
top40_slow_ftp = (
    analyze_top40_conflict_pair(
        9,
        11,
    )
)

top40_benign_infil = (
    analyze_top40_conflict_pair(
        0,
        12,
    )
)

print(
    "=== Top40 SlowHTTP / FTP ==="
)

print(
    json.dumps(
        top40_slow_ftp,
        indent=2,
    )
)

print(
    "\n=== Top40 Benign / Infilteration ==="
)

print(
    json.dumps(
        top40_benign_infil,
        indent=2,
    )
)

=== Top40 SlowHTTP / FTP ===
{
  "class_a": 9,
  "class_b": 11,
  "conflict_patterns": 10,
  "rows_class_a": 71831,
  "rows_class_b": 157005,
  "total_conflict_rows": 228836,
  "unavoidable_minority_rows": 66638,
  "max_frequency_a": 62024,
  "max_frequency_b": 87297
}

=== Top40 Benign / Infilteration ===
{
  "class_a": 0,
  "class_b": 12,
  "conflict_patterns": 15508,
  "rows_class_a": 90480,
  "rows_class_b": 28919,
  "total_conflict_rows": 119399,
  "unavoidable_minority_rows": 28253,
  "max_frequency_a": 605,
  "max_frequency_b": 30
}


In [46]:
conflict_compare = pd.DataFrame([
    {
        "pair": "SlowHTTP / FTP",
        "features": "Full78",
        **slow_ftp,
    },
    {
        "pair": "SlowHTTP / FTP",
        "features": "Top40",
        **top40_slow_ftp,
    },
    {
        "pair": "Benign / Infilteration",
        "features": "Full78",
        **benign_infil,
    },
    {
        "pair": "Benign / Infilteration",
        "features": "Top40",
        **top40_benign_infil,
    },
])

conflict_compare[
    "class_a_conflict_ratio"
] = (
    conflict_compare[
        "rows_class_a"
    ]
    / conflict_compare[
        "class_a"
    ].map(train_counts)
)

conflict_compare[
    "class_b_conflict_ratio"
] = (
    conflict_compare[
        "rows_class_b"
    ]
    / conflict_compare[
        "class_b"
    ].map(train_counts)
)

conflict_compare[
    "ambiguity_ratio"
] = (
    conflict_compare[
        "unavoidable_minority_rows"
    ]
    / conflict_compare[
        "total_conflict_rows"
    ]
)

print(
    conflict_compare[
        [
            "pair",
            "features",
            "conflict_patterns",
            "rows_class_a",
            "rows_class_b",
            "total_conflict_rows",
            "unavoidable_minority_rows",
            "class_a_conflict_ratio",
            "class_b_conflict_ratio",
            "ambiguity_ratio",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

conflict_compare.to_csv(
    RESULT_DIR
    / "full78_vs_top40_conflict.csv",
    index=False,
)

                  pair features  conflict_patterns  rows_class_a  rows_class_b  total_conflict_rows  unavoidable_minority_rows  class_a_conflict_ratio  class_b_conflict_ratio  ambiguity_ratio
        SlowHTTP / FTP   Full78                 10         71831        157005               228836                      66638                0.840542                0.907108         0.291204
        SlowHTTP / FTP    Top40                 10         71831        157005               228836                      66638                0.840542                0.907108         0.291204
Benign / Infilteration   Full78              15493         90450         28904               119354                      28237                0.009570                0.255537         0.236582
Benign / Infilteration    Top40              15508         90480         28919               119399                      28253                0.009573                0.255669         0.236627


In [47]:
from pathlib import Path
import pandas as pd
import numpy as np
import gc
import time

V4_TRAIN_DIR = (
    V4_DIR
    / "training"
)

V4_TRAIN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

V4A_PATH = (
    V4_TRAIN_DIR
    / "train_v4a.parquet"
)

BENIGN_CLASS_ID = 0
BENIGN_SAMPLE_SIZE = 1_000_000
RANDOM_SEED = 42

print("V4-A output:")
print(V4A_PATH)

V4-A output:
/home/geonug/kdt-linux/ML/data/processed/network_ml_dev_v4/training/train_v4a.parquet


In [48]:
pattern_counts = {}

for class_id in range(15):

    path = (
        TOP40_AGG_DIR
        / f"class_{class_id:02d}.parquet"
    )

    count = con.execute(
        f"""
        SELECT COUNT(*)
        FROM read_parquet('{path}')
        """
    ).fetchone()[0]

    pattern_counts[class_id] = int(count)

pattern_counts_df = pd.DataFrame(
    [
        {
            "label_id": class_id,
            "unique_patterns": count,
        }
        for class_id, count
        in pattern_counts.items()
    ]
)

print(
    pattern_counts_df.to_string(
        index=False
    )
)

print(
    "\nTotal unique patterns:",
    f"{sum(pattern_counts.values()):,}"
)

 label_id  unique_patterns
        0          7445262
        1           100976
        2              394
        3              165
        4           139130
        5             1203
        6           402392
        7            29090
        8           101779
        9               44
       10             6874
       11               39
       12            98237
       13               54
       14            66117

Total unique patterns: 8,391,756


In [49]:
benign_sampling_correction = (
    pattern_counts[BENIGN_CLASS_ID]
    / BENIGN_SAMPLE_SIZE
)

print(
    "Benign unique patterns:",
    f"{pattern_counts[0]:,}"
)

print(
    "Benign sampled:",
    f"{BENIGN_SAMPLE_SIZE:,}"
)

print(
    "Sampling correction:",
    f"{benign_sampling_correction:.6f}"
)

Benign unique patterns: 7,445,262
Benign sampled: 1,000,000
Sampling correction: 7.445262


In [50]:
if V4A_PATH.exists():
    print(
        "[EXISTS]",
        V4A_PATH
    )

else:

    start = time.perf_counter()

    benign_path = (
        TOP40_AGG_DIR
        / "class_00.parquet"
    )

    other_paths = [
        str(
            TOP40_AGG_DIR
            / f"class_{class_id:02d}.parquet"
        )
        for class_id in range(1, 15)
    ]

    other_path_sql = ", ".join(
        f"'{path}'"
        for path in other_paths
    )

    con.execute(
        f"""
        COPY (

            WITH benign_sample AS (

                SELECT
                    {TOP40_SQL},
                    label_id,
                    frequency,

                    SQRT(
                        frequency
                    )
                    * {benign_sampling_correction}
                        AS sample_weight

                FROM read_parquet(
                    '{benign_path}'
                )

                USING SAMPLE
                    {BENIGN_SAMPLE_SIZE}
                    ROWS
                    (
                        reservoir,
                        {RANDOM_SEED}
                    )
            ),

            other_classes AS (

                SELECT
                    {TOP40_SQL},
                    label_id,
                    frequency,

                    SQRT(
                        frequency
                    ) AS sample_weight

                FROM read_parquet(
                    [{other_path_sql}]
                )
            )

            SELECT *
            FROM benign_sample

            UNION ALL

            SELECT *
            FROM other_classes

        )
        TO '{V4A_PATH}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )

    elapsed = (
        time.perf_counter()
        - start
    )

    print(
        "[DONE]",
        V4A_PATH
    )

    print(
        "Elapsed:",
        f"{elapsed:.1f}s"
    )

    print(
        "Size:",
        f"{V4A_PATH.stat().st_size / 1024**2:.2f} MB"
    )

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[DONE] /home/geonug/kdt-linux/ML/data/processed/network_ml_dev_v4/training/train_v4a.parquet
Elapsed: 16.3s
Size: 111.45 MB


In [51]:
v4a_expected_rows = (
    BENIGN_SAMPLE_SIZE
    + sum(
        pattern_counts[class_id]
        for class_id in range(1, 15)
    )
)

v4a_rows = con.execute(
    f"""
    SELECT COUNT(*)
    FROM read_parquet(
        '{V4A_PATH}'
    )
    """
).fetchone()[0]

print(
    "Expected:",
    f"{v4a_expected_rows:,}"
)

print(
    "Actual:",
    f"{v4a_rows:,}"
)

assert (
    v4a_rows
    == v4a_expected_rows
)

Expected: 1,946,494
Actual: 1,946,494


In [52]:
v4a_distribution = con.execute(
    f"""
    SELECT
        label_id,

        COUNT(*) AS sampled_patterns,

        SUM(frequency)
            AS represented_frequency,

        SUM(sample_weight)
            AS effective_weight,

        MIN(sample_weight)
            AS min_weight,

        MAX(sample_weight)
            AS max_weight,

        AVG(sample_weight)
            AS avg_weight

    FROM read_parquet(
        '{V4A_PATH}'
    )

    GROUP BY label_id

    ORDER BY label_id
    """
).df()

print(
    v4a_distribution.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

 label_id  sampled_patterns  represented_frequency  effective_weight  min_weight  max_weight  avg_weight
        0           1000000         1266378.000000    7836907.438673    7.445262  616.699640    7.836907
        1            100976          194856.000000     106081.543050    1.000000   40.373258    1.050562
        2               394             428.000000        407.084217    1.000000    2.236068    1.033209
        3               165             166.000000        165.414214    1.000000    1.414214    1.002510
        4            139130          475783.000000     194053.409463    1.000000   12.727922    1.394763
        5              1203            1203.000000       1203.000000    1.000000    1.000000    1.000000
        6            402392          402962.000000     402628.005354    1.000000    1.732051    1.000587
        7             29090           29158.000000      29117.973770    1.000000    1.732051    1.000962
        8            101779          324733.000000     

In [53]:
total_effective_weight = (
    v4a_distribution[
        "effective_weight"
    ].sum()
)

v4a_distribution[
    "effective_ratio"
] = (
    v4a_distribution[
        "effective_weight"
    ]
    / total_effective_weight
)

print(
    v4a_distribution[
        [
            "label_id",
            "sampled_patterns",
            "effective_weight",
            "effective_ratio",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

 label_id  sampled_patterns  effective_weight  effective_ratio
        0           1000000    7836907.438673         0.880661
        1            100976     106081.543050         0.011921
        2               394        407.084217         0.000046
        3               165        165.414214         0.000019
        4            139130     194053.409463         0.021806
        5              1203       1203.000000         0.000135
        6            402392     402628.005354         0.045245
        7             29090      29117.973770         0.003272
        8            101779     149539.841536         0.016804
        9                44        948.437168         0.000107
       10              6874       6997.509753         0.000786
       11                39       1148.626980         0.000129
       12             98237     102807.933538         0.011553
       13                54         54.414214         0.000006
       14             66117      66835.056870         0

In [54]:
def get_effective_weight(
    class_id
):
    return float(
        v4a_distribution.loc[
            v4a_distribution[
                "label_id"
            ] == class_id,
            "effective_weight",
        ].iloc[0]
    )


benign_weight = (
    get_effective_weight(0)
)

infil_weight = (
    get_effective_weight(12)
)

slow_weight = (
    get_effective_weight(9)
)

ftp_weight = (
    get_effective_weight(11)
)


print(
    "Benign : Infilteration =",
    f"{benign_weight / infil_weight:.2f}:1"
)

print(
    "SlowHTTP : FTP =",
    f"{slow_weight / ftp_weight:.4f}:1"
)

Benign : Infilteration = 76.23:1
SlowHTTP : FTP = 0.8257:1


In [55]:
weight_check = con.execute(
    f"""
    SELECT
        COUNT(*) AS total_rows,

        SUM(
            CASE
                WHEN sample_weight IS NULL
                    THEN 1
                ELSE 0
            END
        ) AS null_weights,

        SUM(
            CASE
                WHEN sample_weight <= 0
                    THEN 1
                ELSE 0
            END
        ) AS non_positive_weights,

        MIN(sample_weight)
            AS min_weight,

        MAX(sample_weight)
            AS max_weight,

        AVG(sample_weight)
            AS avg_weight

    FROM read_parquet(
        '{V4A_PATH}'
    )
    """
).df()

print(
    weight_check.to_string(
        index=False
    )
)

 total_rows  null_weights  non_positive_weights  min_weight  max_weight  avg_weight
    1946494           0.0                   0.0         1.0   616.69964    4.571756


In [57]:
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "packages/network-security-ai/pyproject.toml").is_file())

VALID_V2_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "network_ml_split_v2"
    / "valid.parquet"
)

V4_VALID_DIR = (
    V4_DIR
    / "validation"
)

V4_VALID_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

VALID_TOP40_PATH = (
    V4_VALID_DIR
    / "valid_v2_top40.parquet"
)

print("VALID_V2_PATH:")
print(VALID_V2_PATH)

print("\nExists:")
print(VALID_V2_PATH.exists())

if not VALID_V2_PATH.exists():
    raise FileNotFoundError(
        f"Valid V2 파일을 찾을 수 없습니다: "
        f"{VALID_V2_PATH}"
    )

VALID_V2_PATH:
/home/geonug/kdt-linux/ML/data/processed/network_ml_split_v2/valid.parquet

Exists:
True


In [58]:
if VALID_TOP40_PATH.exists():

    print(
        "[SKIP]",
        VALID_TOP40_PATH
    )

else:

    con.execute(
        f"""
        COPY (
            SELECT
                {TOP40_SQL},
                label_id

            FROM read_parquet(
                '{VALID_V2_PATH}'
            )
        )
        TO '{VALID_TOP40_PATH}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )

    print(
        "[DONE]",
        VALID_TOP40_PATH
    )


print(
    "Size:",
    f"{VALID_TOP40_PATH.stat().st_size / 1024**2:.2f} MB"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[DONE] /home/geonug/kdt-linux/ML/data/processed/network_ml_dev_v4/validation/valid_v2_top40.parquet
Size: 131.47 MB


In [59]:
valid_check = con.execute(
    f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT label_id)
            AS classes

    FROM read_parquet(
        '{VALID_TOP40_PATH}'
    )
    """
).df()

print(
    valid_check.to_string(
        index=False
    )
)

   rows  classes
2390002       15


In [60]:
import lightgbm as lgb
import numpy as np
import pandas as pd
import gc
import time
from pathlib import Path

train_df = pd.read_parquet(
    V4A_PATH,
    columns=(
        TOP40_FEATURES
        + [
            "label_id",
            "sample_weight",
        ]
    ),
)

print("Loaded:", train_df.shape)

X_train = train_df[
    TOP40_FEATURES
].to_numpy(
    dtype=np.float32,
    copy=True,
)

y_train = train_df[
    "label_id"
].to_numpy(
    dtype=np.int16,
    copy=True,
)

w_train = train_df[
    "sample_weight"
].to_numpy(
    dtype=np.float32,
    copy=True,
)

del train_df
gc.collect()

print(
    "X_train:",
    X_train.shape,
    X_train.dtype,
)

print(
    "X_train MB:",
    f"{X_train.nbytes / 1024**2:.2f}",
)

print(
    "y_train MB:",
    f"{y_train.nbytes / 1024**2:.2f}",
)

print(
    "w_train MB:",
    f"{w_train.nbytes / 1024**2:.2f}",
)

print(
    "Weight range:",
    float(w_train.min()),
    "~",
    float(w_train.max()),
)

Loaded: (1946494, 42)
X_train: (1946494, 40) float32
X_train MB: 297.01
y_train MB: 3.71
w_train MB: 7.43
Weight range: 1.0 ~ 616.6996459960938


In [61]:
train_set = lgb.Dataset(
    X_train,
    label=y_train,
    weight=w_train,
    feature_name=TOP40_FEATURES,
    free_raw_data=True,
)

train_set.construct()

print(
    "LightGBM Dataset constructed"
)

del X_train
del y_train
del w_train

gc.collect()

print(
    "Raw NumPy arrays released"
)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
LightGBM Dataset constructed
Raw NumPy arrays released


In [64]:
V4A_PARAMS = {
    "objective": "multiclass",
    "num_class": 15,

    "metric": "multi_logloss",

    "learning_rate": 0.05,

    "num_leaves": 63,
    "max_depth": -1,

    "min_data_in_leaf": 50,

    "feature_fraction": 0.9,
    "bagging_fraction": 0.9,
    "bagging_freq": 1,

    "lambda_l1": 0.0,
    "lambda_l2": 1.0,

    "verbosity": -1,

    "num_threads": 2,

    "seed": 42,
    "feature_fraction_seed": 42,
    "bagging_seed": 42,

    # data_random_seed 제거
}

NUM_BOOST_ROUND = 1000

print(V4A_PARAMS)

{'objective': 'multiclass', 'num_class': 15, 'metric': 'multi_logloss', 'learning_rate': 0.05, 'num_leaves': 63, 'max_depth': -1, 'min_data_in_leaf': 50, 'feature_fraction': 0.9, 'bagging_fraction': 0.9, 'bagging_freq': 1, 'lambda_l1': 0.0, 'lambda_l2': 1.0, 'verbosity': -1, 'num_threads': 2, 'seed': 42, 'feature_fraction_seed': 42, 'bagging_seed': 42}


In [65]:
MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "network"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

V4A_MODEL_PATH = (
    MODEL_DIR
    / "lightgbm_top40_v4a.txt"
)

start = time.perf_counter()

model_v4a = lgb.train(
    V4A_PARAMS,
    train_set,
    num_boost_round=NUM_BOOST_ROUND,
)

train_elapsed = (
    time.perf_counter()
    - start
)

# 가장 먼저 모델 저장
model_v4a.save_model(
    str(V4A_MODEL_PATH)
)

print(
    "Training time:",
    f"{train_elapsed:.1f}s",
)

print(
    "Saved:",
    V4A_MODEL_PATH,
)

print(
    "Model size:",
    f"{V4A_MODEL_PATH.stat().st_size / 1024**2:.2f} MB",
)

Training time: 1321.3s
Saved: /home/geonug/kdt-linux/ML/models/network/lightgbm_top40_v4a.txt
Model size: 47.12 MB


In [66]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import gc
import time

from pathlib import Path
from sklearn.metrics import confusion_matrix


# ============================================================
# 1. 저장된 V4-A 모델 다시 로드
# ============================================================

model_v4a = lgb.Booster(
    model_file=str(V4A_MODEL_PATH)
)

print(
    "Loaded model:",
    V4A_MODEL_PATH
)

print(
    "Features:",
    model_v4a.num_feature()
)

assert model_v4a.num_feature() == 40


# ============================================================
# 2. 결과 디렉터리
# ============================================================

V4A_RESULT_DIR = (
    PROJECT_ROOT
    / "results"
    / "lightgbm_top40_v4a"
)

V4A_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 3. Streaming evaluation
# ============================================================

N_CLASSES = 15
BATCH_SIZE = 100_000

cm_v4a = np.zeros(
    (N_CLASSES, N_CLASSES),
    dtype=np.int64,
)

total_rows = 0
correct_rows = 0


valid_select_sql = ", ".join(
    [
        quote_identifier(feature)
        for feature in TOP40_FEATURES
    ]
    + ["label_id"]
)


reader = con.execute(
    f"""
    SELECT
        {valid_select_sql}

    FROM read_parquet(
        '{VALID_TOP40_PATH}'
    )
    """
).fetch_record_batch(
    rows_per_batch=BATCH_SIZE
)


start = time.perf_counter()


for batch_id, batch in enumerate(
    reader,
    start=1,
):

    batch_df = batch.to_pandas()

    X_batch = (
        batch_df[
            TOP40_FEATURES
        ]
        .to_numpy(
            dtype=np.float32,
            copy=False,
        )
    )

    y_batch = (
        batch_df[
            "label_id"
        ]
        .to_numpy(
            dtype=np.int16,
            copy=False,
        )
    )


    proba = model_v4a.predict(
        X_batch,
        num_threads=2,
    )

    pred = np.argmax(
        proba,
        axis=1,
    ).astype(
        np.int16
    )


    cm_v4a += confusion_matrix(
        y_batch,
        pred,
        labels=np.arange(
            N_CLASSES
        ),
    )


    total_rows += len(
        y_batch
    )

    correct_rows += int(
        np.sum(
            pred == y_batch
        )
    )


    if (
        batch_id == 1
        or batch_id % 5 == 0
    ):

        print(
            f"[batch {batch_id:02d}] "
            f"rows={total_rows:,} "
            f"accuracy="
            f"{correct_rows / total_rows:.6f}"
        )


    del (
        batch_df,
        X_batch,
        y_batch,
        proba,
        pred,
    )

    gc.collect()


eval_elapsed = (
    time.perf_counter()
    - start
)


print(
    "\nValidation rows:",
    f"{total_rows:,}"
)

print(
    "Evaluation time:",
    f"{eval_elapsed:.1f}s"
)


assert total_rows == 2_390_002


# ============================================================
# 4. Metrics from confusion matrix
# ============================================================

tp = np.diag(
    cm_v4a
).astype(
    np.float64
)

support = (
    cm_v4a.sum(
        axis=1
    )
    .astype(
        np.float64
    )
)

predicted = (
    cm_v4a.sum(
        axis=0
    )
    .astype(
        np.float64
    )
)


precision = np.divide(
    tp,
    predicted,
    out=np.zeros_like(tp),
    where=predicted != 0,
)


recall = np.divide(
    tp,
    support,
    out=np.zeros_like(tp),
    where=support != 0,
)


f1 = np.divide(
    2 * precision * recall,
    precision + recall,
    out=np.zeros_like(tp),
    where=(
        precision + recall
    ) != 0,
)


accuracy = (
    tp.sum()
    / support.sum()
)

macro_precision = (
    precision.mean()
)

macro_recall = (
    recall.mean()
)

macro_f1 = (
    f1.mean()
)

weighted_f1 = np.average(
    f1,
    weights=support,
)


print("\n=== V4-A FULL VALIDATION ===")

print(
    f"Accuracy        : {accuracy:.6f}"
)

print(
    f"Macro Precision : {macro_precision:.6f}"
)

print(
    f"Macro Recall    : {macro_recall:.6f}"
)

print(
    f"Macro F1        : {macro_f1:.6f}"
)

print(
    f"Weighted F1     : {weighted_f1:.6f}"
)


# ============================================================
# 5. Per-class
# ============================================================

per_class_v4a = pd.DataFrame({
    "label_id": np.arange(
        N_CLASSES
    ),

    "precision": precision,

    "recall": recall,

    "f1": f1,

    "support": support.astype(
        np.int64
    ),
})


print(
    "\n=== PER CLASS ==="
)

print(
    per_class_v4a.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


# ============================================================
# 6. Misclassification
# ============================================================

errors = []


for true_id in range(
    N_CLASSES
):

    for pred_id in range(
        N_CLASSES
    ):

        if true_id == pred_id:
            continue

        count = int(
            cm_v4a[
                true_id,
                pred_id,
            ]
        )

        if count > 0:

            errors.append({
                "true_id": true_id,
                "pred_id": pred_id,
                "count": count,
            })


errors_df = (
    pd.DataFrame(
        errors
    )
    .sort_values(
        "count",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n=== TOP 20 ERRORS ==="
)

print(
    errors_df.head(
        20
    ).to_string(
        index=False
    )
)


# ============================================================
# 7. Save
# ============================================================

metrics_v4a = pd.DataFrame([
    {
        "model":
            "LightGBM Top40 V4-A",

        "accuracy":
            accuracy,

        "macro_precision":
            macro_precision,

        "macro_recall":
            macro_recall,

        "macro_f1":
            macro_f1,

        "weighted_f1":
            weighted_f1,

        "train_patterns":
            1_946_494,

        "weight_strategy":
            "sqrt_frequency",

        "benign_sampling":
            1_000_000,

        "validation_rows":
            total_rows,
    }
])


metrics_v4a.to_csv(
    V4A_RESULT_DIR
    / "metrics.csv",
    index=False,
)


per_class_v4a.to_csv(
    V4A_RESULT_DIR
    / "per_class_metrics.csv",
    index=False,
)


errors_df.to_csv(
    V4A_RESULT_DIR
    / "misclassifications.csv",
    index=False,
)


pd.DataFrame(
    cm_v4a
).to_csv(
    V4A_RESULT_DIR
    / "confusion_matrix.csv",
    index=False,
)


print(
    "\nSaved:",
    V4A_RESULT_DIR
)

Loaded model: /home/geonug/kdt-linux/ML/models/network/lightgbm_top40_v4a.txt
Features: 40


/tmp/ipykernel_1950404/1324938715.py:82: DeprecationWarning: fetch_record_batch() is deprecated, use to_arrow_reader() instead.
  ).fetch_record_batch(


[batch 01] rows=100,000 accuracy=0.993230


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[batch 05] rows=500,000 accuracy=0.997860
[batch 10] rows=1,000,000 accuracy=0.998882
[batch 15] rows=1,500,000 accuracy=0.999237
[batch 20] rows=2,000,000 accuracy=0.992722

Validation rows: 2,390,002
Evaluation time: 3227.8s

=== V4-A FULL VALIDATION ===
Accuracy        : 0.980710
Macro Precision : 0.811396
Macro Recall    : 0.812646
Macro F1        : 0.792164
Weighted F1     : 0.976229

=== PER CLASS ===
 label_id  precision   recall       f1  support
        0   0.989173 0.997984 0.993559  2012773
        1   0.999792 0.999908 0.999850    43368
        2   0.984127 0.632653 0.770186       98
        3   0.862069 0.892857 0.877193       28
        4   1.000000 1.000000 1.000000   104966
        5   0.854478 0.854478 0.854478      268
        6   0.999492 0.999596 0.999544    86583
        7   0.999351 0.999838 0.999594     6160
        8   0.999971 1.000000 0.999986    69024
        9   0.237065 0.858181 0.371505     6882
       10   0.998759 0.999379 0.999069     1611
       11   0

In [67]:
focus_ids = [0, 9, 11, 12]

focus = per_class_v4a[
    per_class_v4a["label_id"].isin(focus_ids)
]

print("=== FOCUS CLASSES ===")
print(
    focus.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print("\n=== TOP 20 ERRORS ===")
print(
    errors_df.head(20).to_string(
        index=False
    )
)

=== FOCUS CLASSES ===
 label_id  precision   recall       f1  support
        0   0.989173 0.997984 0.993559  2012773
        9   0.237065 0.858181 0.371505     6882
       11   0.000000 0.000000 0.000000    19005
       12   0.400506 0.109207 0.171618    24623

=== TOP 20 ERRORS ===
 true_id  pred_id  count
      12        0  21934
      11        9  18999
       0       12   4025
       9       11    976
       5        6     39
       2        0     34
       6        5     32
       0        1      9
      14        9      7
       0        5      7
      11        0      6
       1        0      4
       0        6      4
       0        7      4
       3        0      3
       6        0      2
      13        3      2
       0        8      2
       0        3      2
       0       10      2


In [68]:
print("\n=== CRITICAL PAIR ERRORS ===")

critical_pairs = errors_df[
    (
        (errors_df["true_id"] == 0)
        & (errors_df["pred_id"] == 12)
    )
    |
    (
        (errors_df["true_id"] == 12)
        & (errors_df["pred_id"] == 0)
    )
    |
    (
        (errors_df["true_id"] == 9)
        & (errors_df["pred_id"] == 11)
    )
    |
    (
        (errors_df["true_id"] == 11)
        & (errors_df["pred_id"] == 9)
    )
]

print(
    critical_pairs.to_string(
        index=False
    )
)


=== CRITICAL PAIR ERRORS ===
 true_id  pred_id  count
      12        0  21934
      11        9  18999
       0       12   4025
       9       11    976


In [69]:
VALID_CONFLICT_DIR = (
    V4_VALID_DIR
    / "conflict_parts"
)

VALID_CONFLICT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

for class_id in [0, 9, 11, 12]:

    output = (
        VALID_CONFLICT_DIR
        / f"class_{class_id:02d}.parquet"
    )

    if output.exists():
        print("[SKIP]", class_id)
        continue

    con.execute(
        f"""
        COPY (
            SELECT
                {TOP40_SQL},

                {class_id}::SMALLINT
                    AS label_id,

                COUNT(*)::BIGINT
                    AS frequency

            FROM read_parquet(
                '{VALID_TOP40_PATH}'
            )

            WHERE label_id = {class_id}

            GROUP BY
                {TOP40_SQL}
        )
        TO '{output}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )

    print(
        "[DONE]",
        class_id,
        f"{output.stat().st_size / 1024**2:.2f} MB"
    )

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[DONE] 0 106.87 MB
[DONE] 9 0.01 MB
[DONE] 11 0.01 MB
[DONE] 12 1.19 MB


In [70]:
def analyze_valid_conflict_pair(
    class_a,
    class_b,
):

    path_a = (
        VALID_CONFLICT_DIR
        / f"class_{class_a:02d}.parquet"
    )

    path_b = (
        VALID_CONFLICT_DIR
        / f"class_{class_b:02d}.parquet"
    )

    result = con.execute(
        f"""
        SELECT
            COUNT(*) AS conflict_patterns,

            COALESCE(
                SUM(a.frequency),
                0
            ) AS rows_class_a,

            COALESCE(
                SUM(b.frequency),
                0
            ) AS rows_class_b,

            COALESCE(
                SUM(
                    a.frequency
                    + b.frequency
                ),
                0
            ) AS total_conflict_rows,

            COALESCE(
                SUM(
                    LEAST(
                        a.frequency,
                        b.frequency
                    )
                ),
                0
            ) AS unavoidable_minority_rows

        FROM read_parquet('{path_a}') a

        INNER JOIN read_parquet('{path_b}') b
            ON {TOP40_JOIN_CONDITION}
        """
    ).fetchone()

    return {
        "class_a": class_a,
        "class_b": class_b,
        "conflict_patterns": int(result[0] or 0),
        "rows_class_a": int(result[1] or 0),
        "rows_class_b": int(result[2] or 0),
        "total_conflict_rows": int(result[3] or 0),
        "unavoidable_minority_rows":
            int(result[4] or 0),
    }


valid_slow_ftp = (
    analyze_valid_conflict_pair(
        9,
        11,
    )
)

valid_benign_infil = (
    analyze_valid_conflict_pair(
        0,
        12,
    )
)

print("=== VALID SlowHTTP / FTP ===")
print(valid_slow_ftp)

print("\n=== VALID Benign / Infilteration ===")
print(valid_benign_infil)

=== VALID SlowHTTP / FTP ===
{'class_a': 9, 'class_b': 11, 'conflict_patterns': 2, 'rows_class_a': 5446, 'rows_class_b': 5449, 'total_conflict_rows': 10895, 'unavoidable_minority_rows': 4458}

=== VALID Benign / Infilteration ===
{'class_a': 0, 'class_b': 12, 'conflict_patterns': 3439, 'rows_class_a': 19125, 'rows_class_b': 6312, 'total_conflict_rows': 25437, 'unavoidable_minority_rows': 6149}


In [71]:
VALID_SUPPORT = {
    0: 2_012_773,
    9: 6_882,
    11: 19_005,
    12: 24_623,
}

for name, result in [
    (
        "SlowHTTP / FTP",
        valid_slow_ftp,
    ),
    (
        "Benign / Infilteration",
        valid_benign_infil,
    ),
]:

    a = result["class_a"]
    b = result["class_b"]

    print(f"\n=== {name} ===")

    print(
        "class A conflict:",
        f"{result['rows_class_a'] / VALID_SUPPORT[a]:.4%}"
    )

    print(
        "class B conflict:",
        f"{result['rows_class_b'] / VALID_SUPPORT[b]:.4%}"
    )

    if result["total_conflict_rows"]:

        print(
            "ambiguity ratio:",
            f"{result['unavoidable_minority_rows'] / result['total_conflict_rows']:.4%}"
        )


=== SlowHTTP / FTP ===
class A conflict: 79.1340%
class B conflict: 28.6714%
ambiguity ratio: 40.9179%

=== Benign / Infilteration ===
class A conflict: 0.9502%
class B conflict: 25.6346%
ambiguity ratio: 24.1734%


In [72]:
V4B1_PATH = (
    V4_TRAIN_DIR
    / "train_v4b1.parquet"
)

INFILTRATION_ID = 12
INFILTRATION_CORRECTION = 2.0

if V4B1_PATH.exists():

    print(
        "[SKIP]",
        V4B1_PATH
    )

else:

    con.execute(
        f"""
        COPY (
            SELECT
                {TOP40_SQL},
                label_id,
                frequency,

                CASE
                    WHEN label_id = {INFILTRATION_ID}
                        THEN sample_weight
                             * {INFILTRATION_CORRECTION}

                    ELSE sample_weight
                END
                    AS sample_weight

            FROM read_parquet(
                '{V4A_PATH}'
            )
        )
        TO '{V4B1_PATH}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )

    print(
        "[DONE]",
        V4B1_PATH
    )

print(
    "Size:",
    f"{V4B1_PATH.stat().st_size / 1024**2:.2f} MB"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[DONE] /home/geonug/kdt-linux/ML/data/processed/network_ml_dev_v4/training/train_v4b1.parquet
Size: 111.58 MB


In [73]:
v4b1_weight_check = con.execute(
    f"""
    SELECT
        label_id,
        COUNT(*) AS patterns,
        SUM(sample_weight)
            AS effective_weight,

        MIN(sample_weight)
            AS min_weight,

        MAX(sample_weight)
            AS max_weight

    FROM read_parquet(
        '{V4B1_PATH}'
    )

    GROUP BY label_id
    ORDER BY label_id
    """
).df()

print(
    v4b1_weight_check[
        v4b1_weight_check[
            "label_id"
        ].isin(
            [0, 9, 11, 12]
        )
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

 label_id  patterns  effective_weight  min_weight  max_weight
        0   1000000    7836907.438673    7.445262  616.699640
        9        44        948.437168    1.000000  249.046180
       11        39       1148.626980    1.000000  295.460657
       12     98237     205615.867077    2.000000   10.954451


In [74]:
import lightgbm as lgb
import numpy as np
import pandas as pd
import gc
import time
from pathlib import Path


# ============================================================
# 1. V4-B1 학습 데이터 로드
# ============================================================

train_b1_df = pd.read_parquet(
    V4B1_PATH,
    columns=(
        TOP40_FEATURES
        + [
            "label_id",
            "sample_weight",
        ]
    ),
)

print(
    "Loaded:",
    train_b1_df.shape
)


X_train_b1 = (
    train_b1_df[
        TOP40_FEATURES
    ]
    .to_numpy(
        dtype=np.float32,
        copy=True,
    )
)

y_train_b1 = (
    train_b1_df[
        "label_id"
    ]
    .to_numpy(
        dtype=np.int16,
        copy=True,
    )
)

w_train_b1 = (
    train_b1_df[
        "sample_weight"
    ]
    .to_numpy(
        dtype=np.float32,
        copy=True,
    )
)


del train_b1_df
gc.collect()


print(
    "X:",
    X_train_b1.shape,
    X_train_b1.dtype,
)

print(
    "X MB:",
    f"{X_train_b1.nbytes / 1024**2:.2f}",
)

print(
    "Weight:",
    float(w_train_b1.min()),
    "~",
    float(w_train_b1.max()),
)

Loaded: (1946494, 42)
X: (1946494, 40) float32
X MB: 297.01
Weight: 1.0 ~ 616.6996459960938


In [75]:
V4B1_PARAMS = {
    "objective": "multiclass",
    "num_class": 15,

    "metric": "multi_logloss",

    "learning_rate": 0.05,

    "num_leaves": 63,
    "max_depth": -1,

    "min_data_in_leaf": 50,

    "feature_fraction": 0.9,
    "bagging_fraction": 0.9,
    "bagging_freq": 1,

    "lambda_l1": 0.0,
    "lambda_l2": 1.0,

    "verbosity": -1,

    "num_threads": 2,

    "seed": 42,
    "feature_fraction_seed": 42,
    "bagging_seed": 42,
}

NUM_BOOST_ROUND = 1000

In [76]:
train_b1_set = lgb.Dataset(
    X_train_b1,
    label=y_train_b1,
    weight=w_train_b1,
    feature_name=TOP40_FEATURES,
    free_raw_data=True,
)

train_b1_set.construct()

print(
    "V4-B1 LightGBM Dataset constructed"
)


del X_train_b1
del y_train_b1
del w_train_b1

gc.collect()

print(
    "Raw NumPy arrays released"
)

V4-B1 LightGBM Dataset constructed
Raw NumPy arrays released


In [77]:
V4B1_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "network"
    / "lightgbm_top40_v4b1.txt"
)


start = time.perf_counter()


model_v4b1 = lgb.train(
    V4B1_PARAMS,
    train_b1_set,
    num_boost_round=NUM_BOOST_ROUND,
)


train_elapsed_b1 = (
    time.perf_counter()
    - start
)


model_v4b1.save_model(
    str(V4B1_MODEL_PATH)
)


print(
    "Training time:",
    f"{train_elapsed_b1:.1f}s"
)

print(
    "Saved:",
    V4B1_MODEL_PATH
)

print(
    "Model size:",
    f"{V4B1_MODEL_PATH.stat().st_size / 1024**2:.2f} MB"
)

Training time: 1259.5s
Saved: /home/geonug/kdt-linux/ML/models/network/lightgbm_top40_v4b1.txt
Model size: 46.76 MB


In [78]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import gc
import time

from pathlib import Path
from sklearn.metrics import confusion_matrix


# ============================================================
# 1. V4-B1 모델
# ============================================================

model_v4b1 = lgb.Booster(
    model_file=str(V4B1_MODEL_PATH)
)

assert model_v4b1.num_feature() == 40

print("Loaded:", V4B1_MODEL_PATH)
print("Features:", model_v4b1.num_feature())


# ============================================================
# 2. 결과 경로
# ============================================================

V4B1_RESULT_DIR = (
    PROJECT_ROOT
    / "results"
    / "lightgbm_top40_v4b1"
)

V4B1_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 3. Full Valid streaming evaluation
# ============================================================

N_CLASSES = 15
BATCH_SIZE = 100_000

cm_b1 = np.zeros(
    (N_CLASSES, N_CLASSES),
    dtype=np.int64,
)

total_rows = 0
correct_rows = 0


valid_select_sql = ", ".join(
    [
        quote_identifier(f)
        for f in TOP40_FEATURES
    ]
    + ["label_id"]
)


reader = con.execute(
    f"""
    SELECT
        {valid_select_sql}
    FROM read_parquet(
        '{VALID_TOP40_PATH}'
    )
    """
).fetch_record_batch(
    rows_per_batch=BATCH_SIZE
)


start = time.perf_counter()


for batch_id, batch in enumerate(
    reader,
    start=1,
):

    batch_df = batch.to_pandas()

    X_batch = batch_df[
        TOP40_FEATURES
    ].to_numpy(
        dtype=np.float32,
        copy=False,
    )

    y_batch = batch_df[
        "label_id"
    ].to_numpy(
        dtype=np.int16,
        copy=False,
    )


    proba = model_v4b1.predict(
        X_batch,
        num_threads=2,
    )

    pred = np.argmax(
        proba,
        axis=1,
    ).astype(np.int16)


    cm_b1 += confusion_matrix(
        y_batch,
        pred,
        labels=np.arange(N_CLASSES),
    )

    total_rows += len(y_batch)

    correct_rows += int(
        np.sum(pred == y_batch)
    )


    if (
        batch_id == 1
        or batch_id % 5 == 0
    ):
        print(
            f"[batch {batch_id:02d}] "
            f"rows={total_rows:,} "
            f"accuracy="
            f"{correct_rows / total_rows:.6f}"
        )


    del (
        batch_df,
        X_batch,
        y_batch,
        proba,
        pred,
    )

    gc.collect()


eval_elapsed = (
    time.perf_counter()
    - start
)


print(
    "\nValidation rows:",
    f"{total_rows:,}"
)

print(
    "Evaluation time:",
    f"{eval_elapsed:.1f}s"
)

assert total_rows == 2_390_002


# ============================================================
# 4. Metrics
# ============================================================

tp = np.diag(
    cm_b1
).astype(np.float64)

support = cm_b1.sum(
    axis=1
).astype(np.float64)

predicted = cm_b1.sum(
    axis=0
).astype(np.float64)


precision = np.divide(
    tp,
    predicted,
    out=np.zeros_like(tp),
    where=predicted != 0,
)

recall = np.divide(
    tp,
    support,
    out=np.zeros_like(tp),
    where=support != 0,
)

f1 = np.divide(
    2 * precision * recall,
    precision + recall,
    out=np.zeros_like(tp),
    where=(precision + recall) != 0,
)


accuracy = (
    tp.sum()
    / support.sum()
)

macro_precision = precision.mean()
macro_recall = recall.mean()
macro_f1 = f1.mean()

weighted_f1 = np.average(
    f1,
    weights=support,
)


print("\n=== V4-B1 FULL VALIDATION ===")

print(f"Accuracy        : {accuracy:.6f}")
print(f"Macro Precision : {macro_precision:.6f}")
print(f"Macro Recall    : {macro_recall:.6f}")
print(f"Macro F1        : {macro_f1:.6f}")
print(f"Weighted F1     : {weighted_f1:.6f}")


# ============================================================
# 5. Per-class
# ============================================================

per_class_b1 = pd.DataFrame({
    "label_id": np.arange(N_CLASSES),
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "support": support.astype(np.int64),
})


print("\n=== FOCUS CLASSES ===")

print(
    per_class_b1[
        per_class_b1[
            "label_id"
        ].isin(
            [0, 9, 11, 12]
        )
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


# ============================================================
# 6. Misclassification
# ============================================================

errors = []

for true_id in range(N_CLASSES):

    for pred_id in range(N_CLASSES):

        if true_id == pred_id:
            continue

        count = int(
            cm_b1[
                true_id,
                pred_id,
            ]
        )

        if count > 0:
            errors.append({
                "true_id": true_id,
                "pred_id": pred_id,
                "count": count,
            })


errors_b1 = (
    pd.DataFrame(errors)
    .sort_values(
        "count",
        ascending=False,
    )
    .reset_index(drop=True)
)


print("\n=== TOP 20 ERRORS ===")

print(
    errors_b1
    .head(20)
    .to_string(index=False)
)


# ============================================================
# 7. 저장
# ============================================================

metrics_b1 = pd.DataFrame([{
    "model": "LightGBM Top40 V4-B1",
    "accuracy": accuracy,
    "macro_precision": macro_precision,
    "macro_recall": macro_recall,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1,
    "train_patterns": 1_946_494,
    "weight_strategy": "sqrt_frequency",
    "infilteration_correction": 2.0,
    "validation_rows": total_rows,
}])


metrics_b1.to_csv(
    V4B1_RESULT_DIR / "metrics.csv",
    index=False,
)

per_class_b1.to_csv(
    V4B1_RESULT_DIR
    / "per_class_metrics.csv",
    index=False,
)

errors_b1.to_csv(
    V4B1_RESULT_DIR
    / "misclassifications.csv",
    index=False,
)

pd.DataFrame(
    cm_b1
).to_csv(
    V4B1_RESULT_DIR
    / "confusion_matrix.csv",
    index=False,
)


print(
    "\nSaved:",
    V4B1_RESULT_DIR
)


# ============================================================
# 8. V4-A vs V4-B1 자동 비교
# ============================================================

metrics_a = pd.read_csv(
    PROJECT_ROOT
    / "results"
    / "lightgbm_top40_v4a"
    / "metrics.csv"
)

per_class_a = pd.read_csv(
    PROJECT_ROOT
    / "results"
    / "lightgbm_top40_v4a"
    / "per_class_metrics.csv"
)


comparison = pd.DataFrame({
    "metric": [
        "accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "weighted_f1",
    ],

    "V4-A": [
        float(metrics_a.iloc[0]["accuracy"]),
        float(metrics_a.iloc[0]["macro_precision"]),
        float(metrics_a.iloc[0]["macro_recall"]),
        float(metrics_a.iloc[0]["macro_f1"]),
        float(metrics_a.iloc[0]["weighted_f1"]),
    ],

    "V4-B1": [
        accuracy,
        macro_precision,
        macro_recall,
        macro_f1,
        weighted_f1,
    ],
})

comparison["delta"] = (
    comparison["V4-B1"]
    - comparison["V4-A"]
)


print("\n=== V4-A vs V4-B1 ===")

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


# ============================================================
# 9. Focus class A/B 비교
# ============================================================

focus_ids = [0, 9, 11, 12]

focus_compare = (
    per_class_a[
        per_class_a["label_id"].isin(
            focus_ids
        )
    ][
        [
            "label_id",
            "precision",
            "recall",
            "f1",
        ]
    ]
    .merge(
        per_class_b1[
            per_class_b1["label_id"].isin(
                focus_ids
            )
        ][
            [
                "label_id",
                "precision",
                "recall",
                "f1",
            ]
        ],
        on="label_id",
        suffixes=(
            "_v4a",
            "_v4b1",
        ),
    )
)


for metric in [
    "precision",
    "recall",
    "f1",
]:

    focus_compare[
        f"{metric}_delta"
    ] = (
        focus_compare[
            f"{metric}_v4b1"
        ]
        - focus_compare[
            f"{metric}_v4a"
        ]
    )


print("\n=== FOCUS CLASS COMPARISON ===")

print(
    focus_compare.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

Loaded: /home/geonug/kdt-linux/ML/models/network/lightgbm_top40_v4b1.txt
Features: 40


/tmp/ipykernel_1950404/430262217.py:74: DeprecationWarning: fetch_record_batch() is deprecated, use to_arrow_reader() instead.
  ).fetch_record_batch(


[batch 01] rows=100,000 accuracy=0.993660


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[batch 05] rows=500,000 accuracy=0.997686
[batch 10] rows=1,000,000 accuracy=0.998808
[batch 15] rows=1,500,000 accuracy=0.999177
[batch 20] rows=2,000,000 accuracy=0.993322

Validation rows: 2,390,002
Evaluation time: 3323.3s

=== V4-B1 FULL VALIDATION ===
Accuracy        : 0.979876
Macro Precision : 0.810269
Macro Recall    : 0.827208
Macro F1        : 0.802962
Weighted F1     : 0.976984

=== FOCUS CLASSES ===
 label_id  precision   recall       f1  support
        0   0.990573 0.995541 0.993051  2012773
        9   0.237075 0.858181 0.371517     6882
       11   0.000000 0.000000 0.000000    19005
       12   0.385387 0.227714 0.286276    24623

=== TOP 20 ERRORS ===
 true_id  pred_id  count
      12        0  19016
      11        9  18999
       0       12   8941
       9       11    976
       6        5     36
       2        0     33
       5        6     32
       0        1      8
      14        9      7
      11        0      6
       0        5      6
       1        0    

In [79]:
V4B2_PATH = (
    V4_TRAIN_DIR
    / "train_v4b2.parquet"
)

INFILTRATION_ID = 12

# B1은 이미 V4-A ×2이므로
# B1 × 1.5 = V4-A ×3
B1_TO_B2_MULTIPLIER = 1.5


if V4B2_PATH.exists():

    print(
        "[SKIP]",
        V4B2_PATH
    )

else:

    con.execute(
        f"""
        COPY (
            SELECT
                {TOP40_SQL},
                label_id,
                frequency,

                CASE
                    WHEN label_id = {INFILTRATION_ID}
                        THEN sample_weight
                             * {B1_TO_B2_MULTIPLIER}

                    ELSE sample_weight
                END AS sample_weight

            FROM read_parquet(
                '{V4B1_PATH}'
            )
        )
        TO '{V4B2_PATH}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )

    print(
        "[DONE]",
        V4B2_PATH
    )


print(
    "Size:",
    f"{V4B2_PATH.stat().st_size / 1024**2:.2f} MB"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[DONE] /home/geonug/kdt-linux/ML/data/processed/network_ml_dev_v4/training/train_v4b2.parquet
Size: 111.61 MB


In [80]:
v4b2_weight_check = con.execute(
    f"""
    SELECT
        label_id,
        COUNT(*) AS patterns,
        SUM(sample_weight)
            AS effective_weight,
        MIN(sample_weight)
            AS min_weight,
        MAX(sample_weight)
            AS max_weight

    FROM read_parquet(
        '{V4B2_PATH}'
    )

    GROUP BY label_id
    ORDER BY label_id
    """
).df()


print(
    v4b2_weight_check[
        v4b2_weight_check[
            "label_id"
        ].isin(
            [0, 9, 11, 12]
        )
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

 label_id  patterns  effective_weight  min_weight  max_weight
        0   1000000    7836907.438673    7.445262  616.699640
        9        44        948.437168    1.000000  249.046180
       11        39       1148.626980    1.000000  295.460657
       12     98237     308423.800615    3.000000   16.431677


In [81]:
import lightgbm as lgb
import numpy as np
import pandas as pd
import gc
import time


train_b2_df = pd.read_parquet(
    V4B2_PATH,
    columns=(
        TOP40_FEATURES
        + [
            "label_id",
            "sample_weight",
        ]
    ),
)


print(
    "Loaded:",
    train_b2_df.shape
)


X_train_b2 = (
    train_b2_df[
        TOP40_FEATURES
    ]
    .to_numpy(
        dtype=np.float32,
        copy=True,
    )
)

y_train_b2 = (
    train_b2_df[
        "label_id"
    ]
    .to_numpy(
        dtype=np.int16,
        copy=True,
    )
)

w_train_b2 = (
    train_b2_df[
        "sample_weight"
    ]
    .to_numpy(
        dtype=np.float32,
        copy=True,
    )
)


del train_b2_df
gc.collect()


print(
    "X:",
    X_train_b2.shape,
    X_train_b2.dtype,
)

print(
    "X MB:",
    f"{X_train_b2.nbytes / 1024**2:.2f}"
)

print(
    "Weight range:",
    float(w_train_b2.min()),
    "~",
    float(w_train_b2.max()),
)

Loaded: (1946494, 42)
X: (1946494, 40) float32
X MB: 297.01
Weight range: 1.0 ~ 616.6996459960938


In [82]:
train_b2_set = lgb.Dataset(
    X_train_b2,
    label=y_train_b2,
    weight=w_train_b2,
    feature_name=TOP40_FEATURES,
    free_raw_data=True,
)

train_b2_set.construct()

print(
    "V4-B2 LightGBM Dataset constructed"
)


del X_train_b2
del y_train_b2
del w_train_b2

gc.collect()

print(
    "Raw NumPy arrays released"
)

V4-B2 LightGBM Dataset constructed
Raw NumPy arrays released


In [83]:
V4B2_PARAMS = {
    "objective": "multiclass",
    "num_class": 15,
    "metric": "multi_logloss",

    "learning_rate": 0.05,

    "num_leaves": 63,
    "max_depth": -1,
    "min_data_in_leaf": 50,

    "feature_fraction": 0.9,
    "bagging_fraction": 0.9,
    "bagging_freq": 1,

    "lambda_l1": 0.0,
    "lambda_l2": 1.0,

    "verbosity": -1,
    "num_threads": 2,

    "seed": 42,
    "feature_fraction_seed": 42,
    "bagging_seed": 42,
}

NUM_BOOST_ROUND = 1000

In [84]:
V4B2_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "network"
    / "lightgbm_top40_v4b2.txt"
)


start = time.perf_counter()


model_v4b2 = lgb.train(
    V4B2_PARAMS,
    train_b2_set,
    num_boost_round=NUM_BOOST_ROUND,
)


train_elapsed_b2 = (
    time.perf_counter()
    - start
)


model_v4b2.save_model(
    str(V4B2_MODEL_PATH)
)


print(
    "Training time:",
    f"{train_elapsed_b2:.1f}s"
)

print(
    "Saved:",
    V4B2_MODEL_PATH
)

print(
    "Model size:",
    f"{V4B2_MODEL_PATH.stat().st_size / 1024**2:.2f} MB"
)

Training time: 1297.2s
Saved: /home/geonug/kdt-linux/ML/models/network/lightgbm_top40_v4b2.txt
Model size: 46.38 MB


In [85]:
row12 = v4b2_weight_check[
    v4b2_weight_check["label_id"] == 12
].iloc[0]

print(row12.to_string())

assert abs(
    row12["effective_weight"] - 308423.800615
) < 1.0

assert abs(
    row12["min_weight"] - 3.0
) < 1e-6

print("V4-B2 weight verification: PASS")

label_id                12.000000
patterns             98237.000000
effective_weight    308423.800615
min_weight               3.000000
max_weight              16.431677
V4-B2 weight verification: PASS


In [86]:
V4B2_RESULT_DIR = (
    PROJECT_ROOT
    / "results"
    / "lightgbm_top40_v4b2"
)

V4B2_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [87]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import gc
import time

from pathlib import Path
from sklearn.metrics import confusion_matrix


# ============================================================
# 0. 설정
# ============================================================

N_CLASSES = 15
BATCH_SIZE = 100_000

V4B2_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "network"
    / "lightgbm_top40_v4b2.txt"
)

V4B2_RESULT_DIR = (
    PROJECT_ROOT
    / "results"
    / "lightgbm_top40_v4b2"
)

V4B2_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 1. V4-B2 모델 로드
# ============================================================

model_v4b2 = lgb.Booster(
    model_file=str(V4B2_MODEL_PATH)
)

print("Loaded:", V4B2_MODEL_PATH)
print("Features:", model_v4b2.num_feature())

assert model_v4b2.num_feature() == 40


# ============================================================
# 2. Confusion Matrix 초기화
# ============================================================

cm_b2 = np.zeros(
    (N_CLASSES, N_CLASSES),
    dtype=np.int64,
)

total_rows = 0
correct_rows = 0


# ============================================================
# 3. Validation SELECT 준비
# ============================================================

valid_select_sql = ", ".join(
    [
        quote_identifier(feature)
        for feature in TOP40_FEATURES
    ]
    + ["label_id"]
)


# ============================================================
# 4. Arrow Streaming Reader
#
# fetch_record_batch()가 deprecated 되었으므로
# 이번에는 to_arrow_reader() 사용
# ============================================================

reader = con.execute(
    f"""
    SELECT
        {valid_select_sql}
    FROM read_parquet(
        '{VALID_TOP40_PATH}'
    )
    """
).to_arrow_reader(
    batch_size=BATCH_SIZE
)


# ============================================================
# 5. Full Valid Streaming Evaluation
# ============================================================

start = time.perf_counter()


for batch_id, batch in enumerate(
    reader,
    start=1,
):

    batch_df = batch.to_pandas()

    X_batch = (
        batch_df[
            TOP40_FEATURES
        ]
        .to_numpy(
            dtype=np.float32,
            copy=False,
        )
    )

    y_batch = (
        batch_df[
            "label_id"
        ]
        .to_numpy(
            dtype=np.int16,
            copy=False,
        )
    )


    # --------------------------------------------------------
    # Predict
    # --------------------------------------------------------

    proba = model_v4b2.predict(
        X_batch,
        num_threads=4,
    )

    pred = np.argmax(
        proba,
        axis=1,
    ).astype(
        np.int16
    )


    # --------------------------------------------------------
    # Confusion Matrix 누적
    # --------------------------------------------------------

    cm_b2 += confusion_matrix(
        y_batch,
        pred,
        labels=np.arange(N_CLASSES),
    )


    total_rows += len(y_batch)

    correct_rows += int(
        np.sum(
            pred == y_batch
        )
    )


    if (
        batch_id == 1
        or batch_id % 5 == 0
    ):

        print(
            f"[batch {batch_id:02d}] "
            f"rows={total_rows:,} "
            f"accuracy="
            f"{correct_rows / total_rows:.6f}"
        )


    # --------------------------------------------------------
    # 메모리 해제
    # --------------------------------------------------------

    del (
        batch_df,
        X_batch,
        y_batch,
        proba,
        pred,
    )

    gc.collect()


eval_elapsed = (
    time.perf_counter()
    - start
)


print()
print(
    "Validation rows:",
    f"{total_rows:,}"
)

print(
    "Evaluation time:",
    f"{eval_elapsed:.1f}s"
)


assert total_rows == 2_390_002


# ============================================================
# 6. Overall Metrics
# ============================================================

tp = np.diag(
    cm_b2
).astype(
    np.float64
)


support = (
    cm_b2.sum(
        axis=1
    )
    .astype(
        np.float64
    )
)


predicted = (
    cm_b2.sum(
        axis=0
    )
    .astype(
        np.float64
    )
)


precision = np.divide(
    tp,
    predicted,
    out=np.zeros_like(tp),
    where=predicted != 0,
)


recall = np.divide(
    tp,
    support,
    out=np.zeros_like(tp),
    where=support != 0,
)


f1 = np.divide(
    2 * precision * recall,
    precision + recall,
    out=np.zeros_like(tp),
    where=(
        precision + recall
    ) != 0,
)


accuracy = (
    tp.sum()
    / support.sum()
)

macro_precision = (
    precision.mean()
)

macro_recall = (
    recall.mean()
)

macro_f1 = (
    f1.mean()
)

weighted_f1 = np.average(
    f1,
    weights=support,
)


print()
print(
    "=== V4-B2 FULL VALIDATION ==="
)

print(
    f"Accuracy        : {accuracy:.6f}"
)

print(
    f"Macro Precision : {macro_precision:.6f}"
)

print(
    f"Macro Recall    : {macro_recall:.6f}"
)

print(
    f"Macro F1        : {macro_f1:.6f}"
)

print(
    f"Weighted F1     : {weighted_f1:.6f}"
)


# ============================================================
# 7. Per-Class Metrics
# ============================================================

per_class_b2 = pd.DataFrame({
    "label_id": np.arange(
        N_CLASSES
    ),

    "precision": precision,

    "recall": recall,

    "f1": f1,

    "support": support.astype(
        np.int64
    ),
})


print()
print(
    "=== PER CLASS ==="
)

print(
    per_class_b2.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


# ============================================================
# 8. Focus Classes
# ============================================================

FOCUS_IDS = [
    0,   # Benign
    9,   # SlowHTTPTest
    11,  # FTP-BruteForce
    12,  # Infilteration
]


print()
print(
    "=== FOCUS CLASSES ==="
)

print(
    per_class_b2[
        per_class_b2[
            "label_id"
        ].isin(
            FOCUS_IDS
        )
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


# ============================================================
# 9. Misclassification 생성
# ============================================================

errors = []


for true_id in range(
    N_CLASSES
):

    for pred_id in range(
        N_CLASSES
    ):

        if true_id == pred_id:
            continue

        count = int(
            cm_b2[
                true_id,
                pred_id,
            ]
        )

        if count > 0:

            errors.append({
                "true_id": true_id,
                "pred_id": pred_id,
                "count": count,
            })


errors_b2 = (
    pd.DataFrame(
        errors
    )
    .sort_values(
        "count",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


print()
print(
    "=== TOP 20 ERRORS ==="
)

print(
    errors_b2.head(
        20
    ).to_string(
        index=False
    )
)


# ============================================================
# 10. Critical Pair Errors
# ============================================================

critical_b2 = errors_b2[
    (
        (
            errors_b2["true_id"] == 0
        )
        &
        (
            errors_b2["pred_id"] == 12
        )
    )
    |
    (
        (
            errors_b2["true_id"] == 12
        )
        &
        (
            errors_b2["pred_id"] == 0
        )
    )
    |
    (
        (
            errors_b2["true_id"] == 9
        )
        &
        (
            errors_b2["pred_id"] == 11
        )
    )
    |
    (
        (
            errors_b2["true_id"] == 11
        )
        &
        (
            errors_b2["pred_id"] == 9
        )
    )
]


print()
print(
    "=== CRITICAL PAIR ERRORS ==="
)

print(
    critical_b2.to_string(
        index=False
    )
)


# ============================================================
# 11. B2 결과 저장
# ============================================================

metrics_b2 = pd.DataFrame([
    {
        "model":
            "LightGBM Top40 V4-B2",

        "accuracy":
            accuracy,

        "macro_precision":
            macro_precision,

        "macro_recall":
            macro_recall,

        "macro_f1":
            macro_f1,

        "weighted_f1":
            weighted_f1,

        "train_patterns":
            1_946_494,

        "weight_strategy":
            "sqrt_frequency",

        "infilteration_correction":
            3.0,

        "validation_rows":
            total_rows,

        "evaluation_seconds":
            eval_elapsed,
    }
])


metrics_b2.to_csv(
    V4B2_RESULT_DIR
    / "metrics.csv",
    index=False,
)


per_class_b2.to_csv(
    V4B2_RESULT_DIR
    / "per_class_metrics.csv",
    index=False,
)


errors_b2.to_csv(
    V4B2_RESULT_DIR
    / "misclassifications.csv",
    index=False,
)


pd.DataFrame(
    cm_b2
).to_csv(
    V4B2_RESULT_DIR
    / "confusion_matrix.csv",
    index=False,
)


print()
print(
    "Saved:",
    V4B2_RESULT_DIR
)


# ============================================================
# 12. 기존 V4-A / V4-B1 결과 읽기
# ============================================================

V4A_RESULT_DIR = (
    PROJECT_ROOT
    / "results"
    / "lightgbm_top40_v4a"
)

V4B1_RESULT_DIR = (
    PROJECT_ROOT
    / "results"
    / "lightgbm_top40_v4b1"
)


metrics_a = pd.read_csv(
    V4A_RESULT_DIR
    / "metrics.csv"
)

metrics_b1 = pd.read_csv(
    V4B1_RESULT_DIR
    / "metrics.csv"
)


per_class_a = pd.read_csv(
    V4A_RESULT_DIR
    / "per_class_metrics.csv"
)

per_class_b1 = pd.read_csv(
    V4B1_RESULT_DIR
    / "per_class_metrics.csv"
)


# ============================================================
# 13. A / B1 / B2 Overall 비교
# ============================================================

overall_compare = pd.DataFrame({
    "metric": [
        "accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "weighted_f1",
    ],

    "V4-A_x1": [
        float(
            metrics_a.iloc[0][
                "accuracy"
            ]
        ),

        float(
            metrics_a.iloc[0][
                "macro_precision"
            ]
        ),

        float(
            metrics_a.iloc[0][
                "macro_recall"
            ]
        ),

        float(
            metrics_a.iloc[0][
                "macro_f1"
            ]
        ),

        float(
            metrics_a.iloc[0][
                "weighted_f1"
            ]
        ),
    ],

    "V4-B1_x2": [
        float(
            metrics_b1.iloc[0][
                "accuracy"
            ]
        ),

        float(
            metrics_b1.iloc[0][
                "macro_precision"
            ]
        ),

        float(
            metrics_b1.iloc[0][
                "macro_recall"
            ]
        ),

        float(
            metrics_b1.iloc[0][
                "macro_f1"
            ]
        ),

        float(
            metrics_b1.iloc[0][
                "weighted_f1"
            ]
        ),
    ],

    "V4-B2_x3": [
        accuracy,
        macro_precision,
        macro_recall,
        macro_f1,
        weighted_f1,
    ],
})


overall_compare[
    "B1_minus_A"
] = (
    overall_compare[
        "V4-B1_x2"
    ]
    - overall_compare[
        "V4-A_x1"
    ]
)


overall_compare[
    "B2_minus_B1"
] = (
    overall_compare[
        "V4-B2_x3"
    ]
    - overall_compare[
        "V4-B1_x2"
    ]
)


print()
print(
    "=== V4-A / B1 / B2 OVERALL ==="
)

print(
    overall_compare.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


# ============================================================
# 14. Focus Class A / B1 / B2 비교
# ============================================================

focus_a = (
    per_class_a[
        per_class_a[
            "label_id"
        ].isin(
            FOCUS_IDS
        )
    ][
        [
            "label_id",
            "precision",
            "recall",
            "f1",
        ]
    ]
    .rename(
        columns={
            "precision":
                "precision_A",

            "recall":
                "recall_A",

            "f1":
                "f1_A",
        }
    )
)


focus_b1 = (
    per_class_b1[
        per_class_b1[
            "label_id"
        ].isin(
            FOCUS_IDS
        )
    ][
        [
            "label_id",
            "precision",
            "recall",
            "f1",
        ]
    ]
    .rename(
        columns={
            "precision":
                "precision_B1",

            "recall":
                "recall_B1",

            "f1":
                "f1_B1",
        }
    )
)


focus_b2 = (
    per_class_b2[
        per_class_b2[
            "label_id"
        ].isin(
            FOCUS_IDS
        )
    ][
        [
            "label_id",
            "precision",
            "recall",
            "f1",
        ]
    ]
    .rename(
        columns={
            "precision":
                "precision_B2",

            "recall":
                "recall_B2",

            "f1":
                "f1_B2",
        }
    )
)


focus_compare = (
    focus_a
    .merge(
        focus_b1,
        on="label_id",
    )
    .merge(
        focus_b2,
        on="label_id",
    )
)


focus_compare[
    "recall_B2_minus_B1"
] = (
    focus_compare[
        "recall_B2"
    ]
    - focus_compare[
        "recall_B1"
    ]
)


focus_compare[
    "f1_B2_minus_B1"
] = (
    focus_compare[
        "f1_B2"
    ]
    - focus_compare[
        "f1_B1"
    ]
)


print()
print(
    "=== FOCUS CLASS A / B1 / B2 ==="
)

print(
    focus_compare.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


# ============================================================
# 15. 비교 결과 저장
# ============================================================

COMPARE_DIR = (
    PROJECT_ROOT
    / "results"
    / "v4_model_comparison"
)

COMPARE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


overall_compare.to_csv(
    COMPARE_DIR
    / "overall_metrics.csv",
    index=False,
)


focus_compare.to_csv(
    COMPARE_DIR
    / "focus_class_metrics.csv",
    index=False,
)


print()
print(
    "Comparison saved:",
    COMPARE_DIR
)

Loaded: /home/geonug/kdt-linux/ML/models/network/lightgbm_top40_v4b2.txt
Features: 40
[batch 01] rows=100,000 accuracy=0.992540


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[batch 05] rows=500,000 accuracy=0.997716
[batch 10] rows=1,000,000 accuracy=0.998757
[batch 15] rows=1,500,000 accuracy=0.999118
[batch 20] rows=2,000,000 accuracy=0.993188

Validation rows: 2,390,002
Evaluation time: 1877.5s

=== V4-B2 FULL VALIDATION ===
Accuracy        : 0.979313
Macro Precision : 0.833441
Macro Recall    : 0.830082
Macro F1        : 0.808655
Weighted F1     : 0.977194

=== PER CLASS ===
 label_id  precision   recall       f1  support
        0   0.991097 0.994112 0.992602  2012773
        1   0.999816 0.999908 0.999862    43368
        2   0.984127 0.632653 0.770186       98
        3   0.961538 0.892857 0.925926       28
        4   1.000000 1.000000 1.000000   104966
        5   0.860902 0.854478 0.857678      268
        6   0.999492 0.999584 0.999538    86583
        7   0.999189 0.999838 0.999513     6160
        8   0.999971 1.000000 0.999986    69024
        9   0.242041 0.862831 0.378036     6882
       10   0.998759 0.998759 0.998759     1611
       11   